<a href="https://colab.research.google.com/github/Nefeli-Apostolou/GM-Project---Fraud-Detection/blob/main/Graph_Mining_Project_cleaned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ethereum Graph Mining Project Notebook

## Abstract

This notebook implements the full experimental pipeline for Ethereum fraud detection on a directed transaction graph. The workflow starts from raw transaction data, cleans and normalizes addresses, constructs node and edge tensors for PyTorch Geometric, engineers node-level structural and flow features, merges externally known scam accounts with internally detected fraud accounts, builds semi-supervised labels, and trains several GraphSAGE-based classifiers. Later sections extend the baseline model with temporal motif counts and Snap ML motif-derived features, including late-fusion architectures where GraphSAGE embeddings are concatenated with motif features before MLP classification.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0. Environment setup

Installs the PyTorch Geometric dependency required for `Data` objects and `SAGEConv` layers.


In [2]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 75.4 MB/s eta 0:00:00


## 1. Imports, reusable utilities, and centralized path configuration

This section contains all package imports, small reusable helper functions, reusable GraphSAGE architectures, shared training/evaluation utilities, and the centralized file system configuration. External inputs are listed separately from generated outputs. All generated artifacts are routed into typed subfolders under `outputs/`, so the main Drive project folder remains clean.


In [3]:
import os
import time
import bisect
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
)

# ---------------------------------------------------------
# Centralized project paths
# ---------------------------------------------------------
# Change PROJECT_DIR or DD_PROJECT_DIR here if the notebook is moved.
# External/input files remain in the project folders where they are expected
# to already exist. Generated outputs are written into typed subfolders under
# PROJECT_DIR / "outputs".

PROJECT_DIR = Path("/content/drive/MyDrive/Graph_Mining_Project")
DD_PROJECT_DIR = Path("/content/drive/MyDrive/DD_Project")

OUTPUT_ROOT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIRS = {
    "preprocessed_data": OUTPUT_ROOT_DIR / "01_preprocessed_data",
    "graph_artifacts": OUTPUT_ROOT_DIR / "02_graph_artifacts",
    "labels": OUTPUT_ROOT_DIR / "03_labels",
    "temporal_motifs": OUTPUT_ROOT_DIR / "04_temporal_motifs",
    "model_checkpoints": OUTPUT_ROOT_DIR / "05_model_checkpoints",
    "predictions": OUTPUT_ROOT_DIR / "06_predictions",
    "threshold_tables": OUTPUT_ROOT_DIR / "07_threshold_tables",
    "diagnostics": OUTPUT_ROOT_DIR / "08_diagnostics",
}

for directory in OUTPUT_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

def as_path(path):
    """Return a string path for compatibility with pandas, torch, and os APIs."""
    return str(path)

# ---------------------------------------------------------
# External inputs: called by the notebook but not generated here
# ---------------------------------------------------------
# These files must already exist before running the notebook.

INPUT_PATHS = {
    "raw_transactions": PROJECT_DIR / "eth_tx_last4days.csv",
    "snap_ml_features": PROJECT_DIR / "snap_ml_nodes_output.csv",
    "infomap_communities": PROJECT_DIR / "infomap_communities.csv",
    "internal_detected_fraud": PROJECT_DIR / "all_detected_fraud_accounts.csv",
    "external_scam_list": DD_PROJECT_DIR / "merged_scams.csv",
}

# ---------------------------------------------------------
# Generated outputs: produced by this notebook
# ---------------------------------------------------------

PREPROCESSED_DATA_PATHS = {
    "cleaned_transactions": OUTPUT_DIRS["preprocessed_data"] / "transactions_cleaned_no_contract_creations.csv",
}

GRAPH_ARTIFACT_PATHS = {
    "basic_graph": OUTPUT_DIRS["graph_artifacts"] / "ethereum_pyg_graph_basic_features.pt",
    "node_mapping": OUTPUT_DIRS["graph_artifacts"] / "ethereum_node_mapping.csv",
    "basic_node_features": OUTPUT_DIRS["graph_artifacts"] / "ethereum_node_basic_features.csv",
    "aggregated_directed_edges": OUTPUT_DIRS["graph_artifacts"] / "ethereum_aggregated_directed_edges.csv",
}

FRAUD_LABEL_PATHS = {
    "known_scams_in_raw_transactions": OUTPUT_DIRS["labels"] / "known_scam_addresses_present_in_raw_transactions.csv",
    "known_scams_in_graph": OUTPUT_DIRS["labels"] / "known_scam_addresses_present_in_graph.csv",
    "merged_fraud_addresses": OUTPUT_DIRS["labels"] / "fraud_addresses_merged_internal_and_external.csv",
    "baseline_labels": OUTPUT_DIRS["labels"] / "gnn_labels_baseline_sampled_normals.csv",
    "community_aware_labels": OUTPUT_DIRS["labels"] / "gnn_labels_community_aware_negatives.csv",
}

TEMPORAL_MOTIF_PATHS = {
    "motif_counts_1h_all_nodes": OUTPUT_DIRS["temporal_motifs"] / "temporal_motif_counts_1h_all_nodes.csv",
    "node_features_with_temporal_motifs": OUTPUT_DIRS["temporal_motifs"] / "ethereum_node_features_with_temporal_motifs.csv",
    "graph_with_temporal_motifs": OUTPUT_DIRS["temporal_motifs"] / "ethereum_pyg_graph_with_temporal_motifs.pt",
}

MODEL_CHECKPOINT_PATHS = {
    "basic_1hidden": OUTPUT_DIRS["model_checkpoints"] / "best_graphsage_basic_features_1hidden.pt",
    "basic_threshold_tuned": OUTPUT_DIRS["model_checkpoints"] / "best_graphsage_basic_features_threshold_tuned.pt",
    "community_aware": OUTPUT_DIRS["model_checkpoints"] / "best_graphsage_basic_features_community_aware_labels.pt",
    "temporal_motif_1hidden": OUTPUT_DIRS["model_checkpoints"] / "best_graphsage_temporal_motif_features_1hidden.pt",
    "temporal_motif_2hidden": OUTPUT_DIRS["model_checkpoints"] / "best_graphsage_temporal_motif_features_2hidden.pt",
    "snap_only_1hidden": OUTPUT_DIRS["model_checkpoints"] / "best_graphsage_snap_features_only_1hidden.pt",
    "snap_only_2hidden": OUTPUT_DIRS["model_checkpoints"] / "best_graphsage_snap_features_only_2hidden.pt",
    "temporal_late_fusion": OUTPUT_DIRS["model_checkpoints"] / "best_graphsage_temporal_motif_late_fusion_mlp.pt",
    "snap_fusion": OUTPUT_DIRS["model_checkpoints"] / "best_graphsage_encoder_snap_motif_mlp.pt",
}

PREDICTION_OUTPUT_PATHS = {
    "basic_1hidden": OUTPUT_DIRS["predictions"] / "graphsage_basic_features_predictions.csv",
    "basic_threshold_tuned": OUTPUT_DIRS["predictions"] / "graphsage_basic_features_threshold_tuned_predictions.csv",
    "community_aware": OUTPUT_DIRS["predictions"] / "graphsage_basic_features_community_aware_labels_predictions.csv",
    "temporal_motif_1hidden": OUTPUT_DIRS["predictions"] / "graphsage_temporal_motif_features_1hidden_predictions.csv",
    "temporal_motif_2hidden": OUTPUT_DIRS["predictions"] / "graphsage_temporal_motif_features_2hidden_predictions.csv",
    "snap_only_1hidden": OUTPUT_DIRS["predictions"] / "graphsage_snap_features_only_1hidden_predictions.csv",
    "snap_only_2hidden": OUTPUT_DIRS["predictions"] / "graphsage_snap_features_only_2hidden_predictions.csv",
    "temporal_late_fusion": OUTPUT_DIRS["predictions"] / "graphsage_temporal_motif_late_fusion_mlp_predictions.csv",
    "snap_fusion": OUTPUT_DIRS["predictions"] / "graphsage_encoder_snap_motif_mlp_predictions.csv",
}

THRESHOLD_TABLE_PATHS = {
    "basic_threshold_tuned": OUTPUT_DIRS["threshold_tables"] / "graphsage_basic_features_threshold_tuned_validation_thresholds.csv",
    "community_aware": OUTPUT_DIRS["threshold_tables"] / "graphsage_basic_features_community_aware_validation_thresholds.csv",
    "temporal_motif_1hidden": OUTPUT_DIRS["threshold_tables"] / "graphsage_temporal_motif_features_1hidden_validation_thresholds.csv",
    "temporal_motif_2hidden": OUTPUT_DIRS["threshold_tables"] / "graphsage_temporal_motif_features_2hidden_validation_thresholds.csv",
    "temporal_late_fusion": OUTPUT_DIRS["threshold_tables"] / "graphsage_temporal_motif_late_fusion_validation_thresholds.csv",
}

DIAGNOSTIC_PATHS = {
    "temporal_cycle3_estimation": OUTPUT_DIRS["diagnostics"] / "temporal_cycle3_estimation_diagnostics.csv",
}

# Backward-compatible canonical aliases used by the existing code cells.
PROJECT_DIR = as_path(PROJECT_DIR)
DD_PROJECT_DIR = as_path(DD_PROJECT_DIR)
OUTPUT_ROOT_DIR = as_path(OUTPUT_ROOT_DIR)
GRAPH_OUTPUT_DIR = as_path(OUTPUT_DIRS["graph_artifacts"])
OUTPUT_DIR = as_path(OUTPUT_DIRS["diagnostics"])

CSV_PATH = as_path(INPUT_PATHS["raw_transactions"])
RAW_TX_PATH = CSV_PATH
SNAP_ML_FEATURE_PATH = as_path(INPUT_PATHS["snap_ml_features"])
INFOMAP_COMMUNITIES_PATH = as_path(INPUT_PATHS["infomap_communities"])
INTERNAL_DETECTED_FRAUD_PATH = as_path(INPUT_PATHS["internal_detected_fraud"])
EXTERNAL_SCAM_LIST_PATH = as_path(INPUT_PATHS["external_scam_list"])

CLEANED_TX_PATH = as_path(PREPROCESSED_DATA_PATHS["cleaned_transactions"])

BASIC_GRAPH_PATH = as_path(GRAPH_ARTIFACT_PATHS["basic_graph"])
NODE_MAPPING_FILE_PATH = as_path(GRAPH_ARTIFACT_PATHS["node_mapping"])
NODE_BASIC_FEATURES_PATH = as_path(GRAPH_ARTIFACT_PATHS["basic_node_features"])
AGG_DIRECTED_EDGES_PATH = as_path(GRAPH_ARTIFACT_PATHS["aggregated_directed_edges"])

KNOWN_SCAMS_IN_RAW_TX_PATH = as_path(FRAUD_LABEL_PATHS["known_scams_in_raw_transactions"])
KNOWN_SCAMS_IN_GRAPH_PATH = as_path(FRAUD_LABEL_PATHS["known_scams_in_graph"])
MERGED_FRAUD_ADDRESSES_PATH = as_path(FRAUD_LABEL_PATHS["merged_fraud_addresses"])
BASELINE_LABELS_PATH = as_path(FRAUD_LABEL_PATHS["baseline_labels"])
COMMUNITY_AWARE_LABELS_PATH = as_path(FRAUD_LABEL_PATHS["community_aware_labels"])

TEMPORAL_MOTIF_COUNTS_PATH = as_path(TEMPORAL_MOTIF_PATHS["motif_counts_1h_all_nodes"])
NODE_FEATURES_WITH_TEMPORAL_MOTIFS_PATH = as_path(TEMPORAL_MOTIF_PATHS["node_features_with_temporal_motifs"])
TEMPORAL_MOTIF_GRAPH_PATH = as_path(TEMPORAL_MOTIF_PATHS["graph_with_temporal_motifs"])

BEST_GRAPHSAGE_BASIC_1H_PATH = as_path(MODEL_CHECKPOINT_PATHS["basic_1hidden"])
BEST_GRAPHSAGE_BASIC_THRESHOLD_TUNED_PATH = as_path(MODEL_CHECKPOINT_PATHS["basic_threshold_tuned"])
BEST_GRAPHSAGE_COMMUNITY_AWARE_PATH = as_path(MODEL_CHECKPOINT_PATHS["community_aware"])
BEST_GRAPHSAGE_TEMPORAL_MOTIF_1H_PATH = as_path(MODEL_CHECKPOINT_PATHS["temporal_motif_1hidden"])
BEST_GRAPHSAGE_TEMPORAL_MOTIF_2H_PATH = as_path(MODEL_CHECKPOINT_PATHS["temporal_motif_2hidden"])
BEST_GRAPHSAGE_SNAP_ONLY_1H_PATH = as_path(MODEL_CHECKPOINT_PATHS["snap_only_1hidden"])
BEST_GRAPHSAGE_SNAP_ONLY_2H_PATH = as_path(MODEL_CHECKPOINT_PATHS["snap_only_2hidden"])
BEST_GRAPHSAGE_TEMPORAL_FUSION_PATH = as_path(MODEL_CHECKPOINT_PATHS["temporal_late_fusion"])
BEST_GRAPHSAGE_SNAP_FUSION_PATH = as_path(MODEL_CHECKPOINT_PATHS["snap_fusion"])

GRAPHSAGE_BASIC_PREDICTIONS_PATH = as_path(PREDICTION_OUTPUT_PATHS["basic_1hidden"])
GRAPHSAGE_BASIC_THRESHOLD_TUNED_PREDICTIONS_PATH = as_path(PREDICTION_OUTPUT_PATHS["basic_threshold_tuned"])
GRAPHSAGE_COMMUNITY_AWARE_PREDICTIONS_PATH = as_path(PREDICTION_OUTPUT_PATHS["community_aware"])
GRAPHSAGE_TEMPORAL_MOTIF_1H_PREDICTIONS_PATH = as_path(PREDICTION_OUTPUT_PATHS["temporal_motif_1hidden"])
GRAPHSAGE_TEMPORAL_MOTIF_2H_PREDICTIONS_PATH = as_path(PREDICTION_OUTPUT_PATHS["temporal_motif_2hidden"])
GRAPHSAGE_SNAP_ONLY_1H_PREDICTIONS_PATH = as_path(PREDICTION_OUTPUT_PATHS["snap_only_1hidden"])
GRAPHSAGE_SNAP_ONLY_2H_PREDICTIONS_PATH = as_path(PREDICTION_OUTPUT_PATHS["snap_only_2hidden"])
GRAPHSAGE_TEMPORAL_FUSION_PREDICTIONS_PATH = as_path(PREDICTION_OUTPUT_PATHS["temporal_late_fusion"])
GRAPHSAGE_SNAP_FUSION_PREDICTIONS_PATH = as_path(PREDICTION_OUTPUT_PATHS["snap_fusion"])

GRAPHSAGE_BASIC_THRESHOLD_TABLE_PATH = as_path(THRESHOLD_TABLE_PATHS["basic_threshold_tuned"])
GRAPHSAGE_COMMUNITY_AWARE_THRESHOLD_TABLE_PATH = as_path(THRESHOLD_TABLE_PATHS["community_aware"])
GRAPHSAGE_TEMPORAL_MOTIF_1H_THRESHOLD_TABLE_PATH = as_path(THRESHOLD_TABLE_PATHS["temporal_motif_1hidden"])
GRAPHSAGE_TEMPORAL_MOTIF_2H_THRESHOLD_TABLE_PATH = as_path(THRESHOLD_TABLE_PATHS["temporal_motif_2hidden"])
GRAPHSAGE_TEMPORAL_FUSION_THRESHOLD_TABLE_PATH = as_path(THRESHOLD_TABLE_PATHS["temporal_late_fusion"])

CYCLE3_DIAGNOSTICS_PATH = as_path(DIAGNOSTIC_PATHS["temporal_cycle3_estimation"])

# ---------------------------------------------------------
# Reusable helpers
# ---------------------------------------------------------

def normalize_address_col(s):
    return (
        s.astype(str)
        .str.lower()
        .str.strip()
    )

class GraphSAGEOneHidden(torch.nn.Module):
    """Two GraphSAGE convolution layers: input -> hidden -> output logits."""
    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels,
        dropout=0.3,
    ):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GraphSAGETwoHidden(torch.nn.Module):
    """Three GraphSAGE convolution layers: input -> hidden -> hidden -> output logits."""
    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels,
        dropout=0.3,
    ):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.conv3 = SAGEConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv3(x, edge_index)
        return x

# Default architecture used by the first GraphSAGE experiments.
GraphSAGE = GraphSAGEOneHidden

# Standard training and evaluation utilities for GraphSAGE experiments that
# consume data.x and data.edge_index. The fusion models below keep separate
# custom functions because their forward pass has two feature inputs.
def train_one_epoch():
    model.train()
    optimizer.zero_grad()

    out = model(data.x, data.edge_index)

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask],
        weight=class_weights,
    )

    loss.backward()
    optimizer.step()

    return loss.item()

@torch.no_grad()
def evaluate(mask):
    model.eval()

    out = model(data.x, data.edge_index)

    logits = out[mask]
    labels = data.y[mask]

    probs = F.softmax(logits, dim=1)[:, 1]
    preds = logits.argmax(dim=1)

    acc = (preds == labels).float().mean().item()

    probs_np = probs.detach().cpu().numpy()
    preds_np = preds.detach().cpu().numpy()
    labels_np = labels.detach().cpu().numpy()

    roc_auc = roc_auc_score(labels_np, probs_np)
    pr_auc = average_precision_score(labels_np, probs_np)

    return acc, roc_auc, pr_auc, preds_np, probs_np, labels_np


## 1.1 Files expected as pre-existing inputs

The notebook calls the following files without generating them internally:

| Logical role | Expected path | Why it is external |
|---|---|---|
| Raw Ethereum transaction table | `/content/drive/MyDrive/Graph_Mining_Project/eth_tx_last4days.csv` | Main raw dataset loaded at the start. |
| Snap ML feature table | `/content/drive/MyDrive/Graph_Mining_Project/snap_ml_nodes_output.csv` | Produced by the separate Snap ML motif/feature pipeline, then consumed by the Snap-only and fusion models. |
| Infomap community table | `/content/drive/MyDrive/Graph_Mining_Project/infomap_communities.csv` | Community-detection output used for community-aware negative sampling. |
| Internally detected fraud accounts | `/content/drive/MyDrive/Graph_Mining_Project/all_detected_fraud_accounts.csv` | Fraud seeds/heuristic detections produced before this notebook section. |
| External scam list | `/content/drive/MyDrive/DD_Project/merged_scams.csv` | External known-scam reference list used for overlap and fraud-label merging. |

All other CSV/PT outputs created by this notebook are now stored under `/content/drive/MyDrive/Graph_Mining_Project/outputs/` in subfolders by artifact type.


## 2. Raw transaction loading and validation

Loads the Ethereum transaction table and verifies that the required transaction columns are present before further preprocessing.


In [ ]:

tx = pd.read_csv(CSV_PATH)

required_cols = [
    "block_number",
    "hash",
    "from_address",
    "to_address",
    "value",
    "block_timestamp",
]

missing = set(required_cols) - set(tx.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")


### 2.1 Address normalization and contract-creation filtering

Lowercases sender and receiver addresses and removes transactions with missing destination addresses, which correspond to contract-creation transactions rather than ordinary directed transfers.


In [ ]:

tx["from_address"] = tx["from_address"].astype(str).str.lower()
tx["to_address"] = tx["to_address"].astype(str).str.lower()

# Remove contract creation transactions
# (transactions with missing destination address)

tx = tx[
    ~tx["to_address"].isin(["nan", "none", "null", ""])
].copy()

# Also remove actual NaN values if present
tx = tx.dropna(subset=["from_address", "to_address"])

print(f"Remaining transactions after removing contract creations: {len(tx):,}")


Remaining transactions after removing contract creations: 6,092,594


### 2.2 Wei-to-ETH conversion

Converts the raw `value` field from wei into ETH and stores it as `value_eth` for feature construction.


In [ ]:

tx["value"] = pd.to_numeric(tx["value"], errors="coerce").fillna(0.0)
tx["value_eth"] = tx["value"] / 1e18


### 2.3 Timestamp conversion and cleaned transaction export

Parses timestamps, falls back to Unix-second conversion when necessary, creates a numeric Unix timestamp column, and saves the cleaned transaction table.


In [ ]:
tx["block_timestamp"] = pd.to_datetime(tx["block_timestamp"], errors="coerce")

# fallback if timestamp is Unix seconds
if tx["block_timestamp"].isna().mean() > 0.5:
    tx["block_timestamp"] = pd.to_datetime(
        tx["block_timestamp"],
        unit="s",
        errors="coerce"
    )

tx = tx.dropna(subset=["block_timestamp"])

tx["timestamp_unix"] = tx["block_timestamp"].astype("int64") // 10**9

tx.to_csv(CLEANED_TX_PATH, index=False)

## 3. Node indexing

Creates a deterministic mapping from Ethereum addresses to integer node IDs and appends source/destination node IDs to the transaction table.


In [ ]:

all_addresses = pd.concat(
    [tx["from_address"], tx["to_address"]],
    ignore_index=True
).drop_duplicates()

node_id = pd.Series(
    data=np.arange(len(all_addresses), dtype=np.int64),
    index=all_addresses.values
)

tx["src"] = tx["from_address"].map(node_id).astype(np.int64)
tx["dst"] = tx["to_address"].map(node_id).astype(np.int64)

num_nodes = len(node_id)

print(f"Number of nodes: {num_nodes:,}")
print(f"Number of directed edges: {len(tx):,}")


Number of nodes: 1,828,910
Number of directed edges: 6,092,594


### 3.1 PyTorch Geometric edge tensors

Constructs `edge_index` from directed transfers and stores edge attributes containing ETH value and transaction timestamp.


In [ ]:

edge_index = torch.tensor(
    tx[["src", "dst"]].values.T,
    dtype=torch.long
)

# Edge attributes: value_eth and timestamp only
edge_attr_np = tx[["value_eth", "timestamp_unix"]].copy()
edge_attr_np = edge_attr_np.replace([np.inf, -np.inf], np.nan).fillna(0.0)

edge_attr = torch.tensor(
    edge_attr_np.values,
    dtype=torch.float
)


## 4. Basic node-level flow features

Aggregates outgoing and incoming transaction statistics for each node, including mean and maximum transfer amounts and derived transaction ratios.


In [ ]:

out_stats = tx.groupby("src").agg(
    mean_send_amount=("value_eth", "mean"),
    max_send_amount=("value_eth", "max"),
)

in_stats = tx.groupby("dst").agg(
    mean_recv_amount=("value_eth", "mean"),
    max_recv_amount=("value_eth", "max"),
)

# These are used internally to compute ratios, but dropped later
tmp_out = tx.groupby("src").agg(
    send_num=("hash", "count"),
    send_amount=("value_eth", "sum"),
)

tmp_in = tx.groupby("dst").agg(
    recv_num=("hash", "count"),
    recv_amount=("value_eth", "sum"),
)

node_features = pd.DataFrame(index=np.arange(num_nodes))

node_features = node_features.join(out_stats, how="left")
node_features = node_features.join(in_stats, how="left")
node_features = node_features.join(tmp_out, how="left")
node_features = node_features.join(tmp_in, how="left")
node_features = node_features.fillna(0.0)

node_features["total_tx"] = (
    node_features["send_num"] + node_features["recv_num"]
)

node_features["total_amount"] = (
    node_features["send_amount"] + node_features["recv_amount"]
)

node_features["out_ratio"] = (
    node_features["send_num"] /
    node_features["total_tx"].replace(0, np.nan)
).fillna(0.0)

node_features["in_ratio"] = (
    node_features["recv_num"] /
    node_features["total_tx"].replace(0, np.nan)
).fillna(0.0)

node_features["amount_out_ratio"] = (
    node_features["send_amount"] /
    node_features["total_amount"].replace(0, np.nan)
).fillna(0.0)

node_features["amount_in_ratio"] = (
    node_features["recv_amount"] /
    node_features["total_amount"].replace(0, np.nan)
).fillna(0.0)


### 4.1 Structural graph features

Aggregates repeated directed transactions and computes classical structural features such as PageRank, degree-based features, betweenness, eigenvector centrality, and clustering coefficients.


In [ ]:

# We aggregate repeated transactions between the same pair of addresses.
# This makes centrality computation much cheaper.

edge_weights = (
    tx.groupby(["src", "dst"])
    .agg(
        tx_count=("hash", "count"),
        total_value=("value_eth", "sum")
    )
    .reset_index()
)

print(f"Aggregated directed edges: {len(edge_weights):,}")

# Directed graph for PageRank and directed degrees
G_dir = nx.DiGraph()
G_dir.add_nodes_from(range(num_nodes))
G_dir.add_weighted_edges_from(
    edge_weights[["src", "dst", "tx_count"]].itertuples(index=False, name=None),
    weight="weight"
)

# Undirected graph for clustering, eigenvector, betweenness
G_undir = nx.Graph()
G_undir.add_nodes_from(range(num_nodes))
G_undir.add_weighted_edges_from(
    edge_weights[["src", "dst", "tx_count"]].itertuples(index=False, name=None),
    weight="weight"
)

# Directed unique-neighbor degrees
in_degree_dict = dict(G_dir.in_degree())
out_degree_dict = dict(G_dir.out_degree())

node_features["in_degree"] = pd.Series(in_degree_dict)
node_features["out_degree"] = pd.Series(out_degree_dict)

# PageRank
print("Computing PageRank...")
pagerank_dict = nx.pagerank(
    G_dir,
    alpha=0.85,
    max_iter=100,
    tol=1e-06,
    weight="weight"
)

node_features["pagerank"] = pd.Series(pagerank_dict)

'''
# Clustering coefficient
print("Computing clustering coefficient...")
clustering_dict = nx.clustering(
    G_undir,
    weight="weight"
)

node_features["clustering_coefficient"] = pd.Series(clustering_dict)


# Eigenvector centrality
# This can be slow on very large graphs.
print("Computing eigenvector centrality...")
try:
    eigen_dict = nx.eigenvector_centrality(
        G_undir,
        max_iter=300,
        tol=1e-06,
        weight="weight"
    )
except nx.PowerIterationFailedConvergence:
    print("Eigenvector centrality did not converge; filling with zeros.")
    eigen_dict = {i: 0.0 for i in range(num_nodes)}

node_features["eigenvector_centrality"] = pd.Series(eigen_dict)
'''

# Approximate betweenness centrality
# Exact betweenness is usually impossible on million-node graphs.
# Increase k for better accuracy, decrease k for speed.
print("Computing approximate betweenness centrality...")
BETWEENNESS_SAMPLE_SIZE = min(1000, num_nodes)

betweenness_dict = nx.betweenness_centrality(
    G_undir,
    k=BETWEENNESS_SAMPLE_SIZE,
    normalized=True,
    weight=None,
    seed=42
)

node_features["betweenness"] = pd.Series(betweenness_dict)

node_features = node_features.fillna(0.0)


Aggregated directed edges: 2,948,955
Computing PageRank...
Computing approximate betweenness centrality...


### 4.2 Temporal activity features

Computes node-level temporal activity summaries from transaction timestamps, including transaction timing patterns derived from outgoing and incoming events.


In [ ]:

events_out = tx[["src", "timestamp_unix", "block_timestamp"]].copy()
events_out = events_out.rename(columns={"src": "node"})

events_in = tx[["dst", "timestamp_unix", "block_timestamp"]].copy()
events_in = events_in.rename(columns={"dst": "node"})

events = pd.concat([events_out, events_in], ignore_index=True)
events = events.sort_values(["node", "timestamp_unix"])

events["node_inter_tx_time"] = (
    events.groupby("node")["timestamp_unix"].diff()
)

events["node_inter_tx_time"] = (
    events["node_inter_tx_time"]
    .replace([np.inf, -np.inf], np.nan)
)

inter_stats = events.groupby("node")["node_inter_tx_time"].agg(
    mean_inter_tx_time="mean",
    std_inter_tx_time="std"
).fillna(0.0)

node_features = node_features.join(inter_stats, how="left")
node_features = node_features.fillna(0.0)

mu = node_features["mean_inter_tx_time"]
sigma = node_features["std_inter_tx_time"]

node_features["burstiness"] = (
    (sigma - mu) /
    (sigma + mu).replace(0, np.nan)
).fillna(0.0)

time_span_stats = events.groupby("node")["timestamp_unix"].agg(
    first_tx_time="min",
    last_tx_time="max",
    tx_event_count="count"
)

time_span_stats["active_seconds"] = (
    time_span_stats["last_tx_time"] - time_span_stats["first_tx_time"]
)

time_span_stats["active_days"] = (
    time_span_stats["active_seconds"] / 86400
)

active_hours = (time_span_stats["active_seconds"] / 3600).clip(lower=1)

time_span_stats["tx_per_hour"] = (
    time_span_stats["tx_event_count"] / active_hours
)

node_features = node_features.join(
    time_span_stats[
        [
            "active_days",
            "tx_per_hour",
        ]
    ],
    how="left"
)

node_features = node_features.fillna(0.0)

# Night activity: 00:00–06:00 UTC
events["hour"] = events["block_timestamp"].dt.hour
events["is_night"] = events["hour"].between(0, 5).astype(float)

night_stats = events.groupby("node")["is_night"].mean()
night_stats.name = "night_activity_ratio"

node_features = node_features.join(night_stats, how="left")
node_features["night_activity_ratio"] = (
    node_features["night_activity_ratio"].fillna(0.0)
)


### 4.3 Feature selection

Drops intermediate or redundant columns so that the exported node feature matrix contains the intended model inputs.


In [ ]:

drop_cols = [
    "send_num",
    "recv_num",
    "send_amount",
    "recv_amount",
]

node_features = node_features.drop(columns=drop_cols, errors="ignore")


### 4.4 Feature cleaning and scaling preparation

Replaces invalid numerical values and prepares the node feature matrix for graph construction and model training.


In [ ]:

node_features = node_features.replace([np.inf, -np.inf], 0.0)
node_features = node_features.fillna(0.0)

# Do not log-transform ratios, burstiness, PageRank, clustering, eigenvector, betweenness.
log_cols = [
    "mean_send_amount",
    "mean_recv_amount",
    "max_send_amount",
    "max_recv_amount",
    "total_tx",
    "total_amount",
    "in_degree",
    "out_degree",
    "mean_inter_tx_time",
    "std_inter_tx_time",
    "active_days",
    "tx_per_hour",
]

for col in log_cols:
    if col in node_features.columns:
        node_features[col] = np.log1p(node_features[col].clip(lower=0))

x = torch.tensor(
    node_features.values,
    dtype=torch.float
)

print("Node feature matrix shape:", x.shape)
print("Node features:")
print(node_features.columns.tolist())

Node feature matrix shape: torch.Size([1828910, 20])
Node features:
['mean_send_amount', 'max_send_amount', 'mean_recv_amount', 'max_recv_amount', 'total_tx', 'total_amount', 'out_ratio', 'in_ratio', 'amount_out_ratio', 'amount_in_ratio', 'in_degree', 'out_degree', 'pagerank', 'betweenness', 'mean_inter_tx_time', 'std_inter_tx_time', 'burstiness', 'active_days', 'tx_per_hour', 'night_activity_ratio']


## 5. PyTorch Geometric graph object

Builds the full `Data` object with node features, directed edges, and edge attributes. Labels are added later because the graph is used in semi-supervised experiments.


In [ ]:

data = Data(
    x=x,
    edge_index=edge_index,
    edge_attr=edge_attr,
    num_nodes=num_nodes
)

print(data)

print("\nNode feature matrix shape:")
print(data.x.shape)

print("\nEdge index shape:")
print(data.edge_index.shape)

print("\nEdge attribute shape:")
print(data.edge_attr.shape)


Data(x=[1828910, 20], edge_index=[2, 6092594], edge_attr=[6092594, 2], num_nodes=1828910)

Node feature matrix shape:
torch.Size([1828910, 20])

Edge index shape:
torch.Size([2, 6092594])

Edge attribute shape:
torch.Size([6092594, 2])


### 5.1 Persist graph artifacts

Saves the PyTorch Geometric graph, address-to-node mapping, engineered node features, and aggregated edge list. These files are reused by all subsequent label-building and model-training sections.


In [ ]:

# GRAPH_OUTPUT_DIR is defined and created in the centralized configuration cell.

# Main PyTorch Geometric graph
torch.save(
    data,
    BASIC_GRAPH_PATH
)

# Address ↔ node ID mapping
node_mapping_df = pd.DataFrame({
    "address": node_id.index,
    "node_id": node_id.values
})

node_mapping_df.to_csv(
    NODE_MAPPING_FILE_PATH,
    index=False
)

# Human-readable node features
node_features_export = node_features.copy()
node_features_export["node_id"] = node_features_export.index

node_features_export.to_csv(
    NODE_BASIC_FEATURES_PATH,
    index=False
)

# Aggregated edge list
edge_weights.to_csv(
    AGG_DIRECTED_EDGES_PATH,
    index=False
)

print("\nSaved files:")
print(BASIC_GRAPH_PATH)
print(NODE_MAPPING_FILE_PATH)
print(NODE_BASIC_FEATURES_PATH)
print(AGG_DIRECTED_EDGES_PATH)

# Aggregated edge list:
# Each row represents a unique directed connection between two addresses.
#
# Columns:
# - src: source node ID
# - dst: destination node ID
# - tx_count: number of transactions between the two nodes
# - total_value: total ETH transferred across those transactions
#
# This compressed graph is useful for:
# - temporal motif extraction
# - community detection
# - graph visualization
# - debugging and inspection
# - temporal aggregation
# - classical network analysis
#
# Using aggregated edges is much more memory-efficient than repeatedly
# processing the raw transaction table.


Saved files:
/content/drive/MyDrive/Graph_Mining_Project/outputs/02_graph_artifacts/ethereum_pyg_graph_basic_features.pt
/content/drive/MyDrive/Graph_Mining_Project/outputs/02_graph_artifacts/ethereum_node_mapping.csv
/content/drive/MyDrive/Graph_Mining_Project/outputs/02_graph_artifacts/ethereum_node_basic_features.csv
/content/drive/MyDrive/Graph_Mining_Project/outputs/02_graph_artifacts/ethereum_aggregated_directed_edges.csv


## 6. External scam-list overlap analysis

Loads an external scam-address list we collected, normalizes addresses, checks overlap with the raw transaction dataset and the constructed graph, and identifies counterparties that interacted with known scam accounts. This was used by us to understand how big our available group of known fraudulent accounts was in the graph.


In [ ]:

# Canonical input/output paths are defined in the centralized configuration cell.

# Address normalization uses the shared normalize_address_col() helper from the configuration cell.

# ---------------------------------------------------------
# 1. Load scam address file
# ---------------------------------------------------------

scams = pd.read_csv(EXTERNAL_SCAM_LIST_PATH)

if "address" not in scams.columns:
    raise ValueError("The scam CSV must contain a column named 'address'.")

scams["address"] = normalize_address_col(scams["address"])

# Remove duplicates because scam accounts may repeat
unique_scam_addresses = set(scams["address"].dropna())

print("Rows in scam file:", len(scams))
print("Unique scam addresses:", len(unique_scam_addresses))

# ---------------------------------------------------------
# 2. Check how many scam accounts are in eth_tx_last4days_2.csv
# ---------------------------------------------------------

tx = pd.read_csv(
    RAW_TX_PATH,
    usecols=["from_address", "to_address"]
)

tx["from_address"] = normalize_address_col(tx["from_address"])
tx["to_address"] = normalize_address_col(tx["to_address"])

# Unique accounts appearing in raw transaction file
tx_accounts = set(tx["from_address"].dropna()) | set(tx["to_address"].dropna())

scams_in_tx = unique_scam_addresses.intersection(tx_accounts)

print("\nRaw transaction CSV:")
print("Unique accounts in eth_tx_last4days_2.csv:", len(tx_accounts))
print("Scam accounts found in transaction CSV:", len(scams_in_tx))
print(
    "Percentage of scam list found in transaction CSV:",
    round(len(scams_in_tx) / len(unique_scam_addresses) * 100, 4),
    "%"
)

# ---------------------------------------------------------
# 3. Check how many scam accounts are in the graph
# ---------------------------------------------------------

node_mapping = pd.read_csv(NODE_MAPPING_FILE_PATH)

node_mapping["address"] = normalize_address_col(node_mapping["address"])

graph_accounts = set(node_mapping["address"].dropna())

scams_in_graph = unique_scam_addresses.intersection(graph_accounts)

print("\nCreated graph:")
print("Unique accounts in graph:", len(graph_accounts))
print("Scam accounts found in graph:", len(scams_in_graph))
print(
    "Percentage of scam list found in graph:",
    round(len(scams_in_graph) / len(unique_scam_addresses) * 100, 4),
    "%"
)


# ---------------------------------------------------------
# 5. Find all nodes that transacted with scam accounts
# ---------------------------------------------------------

# Transactions where either side is a known scam
tx_with_scams = tx[
    tx["from_address"].isin(unique_scam_addresses) |
    tx["to_address"].isin(unique_scam_addresses)
].copy()

# Collect counterparties
counterparties_from = set(
    tx_with_scams.loc[
        tx_with_scams["from_address"].isin(unique_scam_addresses),
        "to_address"
    ].dropna()
)

counterparties_to = set(
    tx_with_scams.loc[
        tx_with_scams["to_address"].isin(unique_scam_addresses),
        "from_address"
    ].dropna()
)

# Union of all counterparties
counterparty_nodes = counterparties_from | counterparties_to

# Remove scam accounts themselves
counterparty_nodes = counterparty_nodes - unique_scam_addresses

print("\nCounterparty Analysis")
print("Unique nodes that interacted with scam accounts:",
      len(counterparty_nodes))

# ---------------------------------------------------------
# 6. Check how many of these counterparties are in graph
# ---------------------------------------------------------

counterparties_in_graph = counterparty_nodes.intersection(graph_accounts)

print("\nCounterparties present in graph:")
print(len(counterparties_in_graph))





Rows in scam file: 8430
Unique scam addresses: 6717

Raw transaction CSV:
Unique accounts in eth_tx_last4days_2.csv: 1829064
Scam accounts found in transaction CSV: 131
Percentage of scam list found in transaction CSV: 1.9503 %

Created graph:
Unique accounts in graph: 1828910
Scam accounts found in graph: 131
Percentage of scam list found in graph: 1.9503 %

Counterparty Analysis
Unique nodes that interacted with scam accounts: 679892

Counterparties present in graph:
679891


### 6.1 Save external scam overlaps

Exports the scam addresses found in the transaction CSV and the subset that also appears in the graph node mapping.


In [ ]:
# ---------------------------------------------------------
# 4. Save matched scam accounts
# ---------------------------------------------------------

scams_in_tx_df = pd.DataFrame({
    "address": sorted(scams_in_tx)
})

scams_in_graph_df = node_mapping[
    node_mapping["address"].isin(scams_in_graph)
].copy()

scams_in_tx_df.to_csv(
    KNOWN_SCAMS_IN_RAW_TX_PATH,
    index=False
)

scams_in_graph_df.to_csv(
    KNOWN_SCAMS_IN_GRAPH_PATH,
    index=False
)

print("\nSaved:")
print(KNOWN_SCAMS_IN_RAW_TX_PATH)
print(KNOWN_SCAMS_IN_GRAPH_PATH)


Saved:
/content/drive/MyDrive/Graph_Mining_Project/outputs/03_labels/known_scam_addresses_present_in_raw_transactions.csv
/content/drive/MyDrive/Graph_Mining_Project/outputs/03_labels/known_scam_addresses_present_in_graph.csv


### 6.2 Merge external scam addresses with internally detected fraud

Combines externally known scam accounts with the project’s internally detected fraud accounts, preserving source labels and deduplicating addresses.


In [ ]:
# ---------------------------------------------------------
# Merge known scam accounts with your detected fraud accounts
# ---------------------------------------------------------

# Canonical input/output paths are defined in the centralized configuration cell.

# ---------------------------------------------------------
# Load files
# ---------------------------------------------------------

scam_overlap = pd.read_csv(KNOWN_SCAMS_IN_RAW_TX_PATH)
detected_fraud = pd.read_csv(INTERNAL_DETECTED_FRAUD_PATH)

# ---------------------------------------------------------
# Normalize addresses
# ---------------------------------------------------------
# Address normalization uses the shared normalize_address_col() helper.

# Rename address column if needed
possible_cols = ["address", "wallet", "wallet_address"]

found_col = None
for col in possible_cols:
    if col in detected_fraud.columns:
        found_col = col
        break

if found_col is None:
    raise ValueError(
        f"Could not find an address column in detected fraud file. "
        f"Columns are: {detected_fraud.columns.tolist()}"
    )

detected_fraud = detected_fraud.rename(
    columns={found_col: "address"}
)

# Normalize
scam_overlap["address"] = normalize_address_col(
    scam_overlap["address"]
)

detected_fraud["address"] = normalize_address_col(
    detected_fraud["address"]
)

# ---------------------------------------------------------
# Add source labels
# ---------------------------------------------------------

scam_overlap["source"] = "online_scam_list"
detected_fraud["source"] = "our_detected_fraud"

# ---------------------------------------------------------
# Merge and deduplicate
# ---------------------------------------------------------

merged_fraud = pd.concat(
    [
        scam_overlap[["address", "source"]],
        detected_fraud[["address", "source"]]
    ],
    ignore_index=True
)

# Keep track of addresses appearing in both sources
merged_fraud = (
    merged_fraud
    .groupby("address")["source"]
    .apply(lambda x: ",".join(sorted(set(x))))
    .reset_index()
)

# ---------------------------------------------------------
# Statistics
# ---------------------------------------------------------

print("Online scam accounts in transactions:",
      len(scam_overlap))

print("Our detected fraud accounts:",
      len(detected_fraud))

print("Unique merged fraud accounts:",
      len(merged_fraud))

# Accounts appearing in both lists
both_sources = merged_fraud[
    merged_fraud["source"].str.contains(",")
]

print("Accounts appearing in BOTH sources:",
      len(both_sources))

# ---------------------------------------------------------
# Save merged fraud list
# ---------------------------------------------------------

OUTPUT_PATH = MERGED_FRAUD_ADDRESSES_PATH

merged_fraud.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved:")
print("- fraud_addresses_merged_internal_and_external.csv")

Online scam accounts in transactions: 131
Our detected fraud accounts: 4351
Unique merged fraud accounts: 4466
Accounts appearing in BOTH sources: 16

Saved:
- fraud_addresses_merged_internal_and_external.csv


## 7. Baseline label construction

Defines the paths required to create a supervised label table from fraud addresses, graph edges, and Infomap community assignments.


In [ ]:

NODE_MAPPING_PATH = NODE_MAPPING_FILE_PATH
AGG_EDGES_PATH = AGG_DIRECTED_EDGES_PATH

# Correct fraud file
FRAUD_PATH = MERGED_FRAUD_ADDRESSES_PATH

# Infomap communities file
INFOMAP_PATH = INFOMAP_COMMUNITIES_PATH

TARGET_COMMUNITY_NORMALS = 25000
TARGET_RANDOM_NORMALS = 20000
RANDOM_SEED = 42

### 7.1 Load graph metadata for labeling

Loads node mappings, aggregated edges, merged fraud addresses, and Infomap community labels.


In [ ]:
# ---------------------------------------------------------
# 1. Load files
# ---------------------------------------------------------

node_mapping = pd.read_csv(NODE_MAPPING_PATH)
edges = pd.read_csv(AGG_EDGES_PATH)
fraud_df = pd.read_csv(FRAUD_PATH)
infomap_df = pd.read_csv(INFOMAP_PATH)

node_mapping["address"] = node_mapping["address"].astype(str).str.lower().str.strip()
infomap_df["address"] = infomap_df["address"].astype(str).str.lower().str.strip()

if "address" in fraud_df.columns:
    fraud_df["address"] = fraud_df["address"].astype(str).str.lower().str.strip()

node_mapping["node_id"] = node_mapping["node_id"].astype(int)
edges["src"] = edges["src"].astype(int)
edges["dst"] = edges["dst"].astype(int)

print("Nodes in graph:", len(node_mapping))
print("Aggregated edges:", len(edges))
print("Merged fraud accounts:", len(fraud_df))
print("Infomap rows:", len(infomap_df))

Nodes in graph: 1828910
Aggregated edges: 2948955
Merged fraud accounts: 4466
Infomap rows: 1686534


### 7.2 Map fraud addresses to graph node IDs

Restricts fraud labels to addresses that are actually present in the constructed graph.


In [ ]:
# ---------------------------------------------------------
# 2. Get fraud node IDs from fraud_addresses_merged_internal_and_external.csv
# ---------------------------------------------------------

if "node_id" in fraud_df.columns:
    fraud_df["node_id"] = pd.to_numeric(fraud_df["node_id"], errors="coerce")
    fraud_df = fraud_df.dropna(subset=["node_id"])
    fraud_df["node_id"] = fraud_df["node_id"].astype(int)

else:
    fraud_df = fraud_df.merge(
        node_mapping[["address", "node_id"]],
        on="address",
        how="inner"
    )

fraud_nodes = set(fraud_df["node_id"].astype(int))

print("Fraud nodes found in graph:", len(fraud_nodes))

Fraud nodes found in graph: 4466


### 7.3 Identify fraud-neighbor exclusions

Builds the set of direct neighbors around fraud nodes so that negative samples can avoid immediate fraud counterparties.


In [ ]:
# ---------------------------------------------------------
# 3. Attach node_id to Infomap communities
# ---------------------------------------------------------

infomap_df = infomap_df.merge(
    node_mapping[["address", "node_id"]],
    on="address",
    how="inner"
)

infomap_df["node_id"] = infomap_df["node_id"].astype(int)

print("Infomap nodes found in graph:", len(infomap_df))

Infomap nodes found in graph: 1686380


### 7.4 Candidate normal nodes from fraud communities

Selects candidate normal nodes from the same Infomap communities as fraud nodes while excluding fraud nodes and their direct neighbors.


In [ ]:
# ---------------------------------------------------------
# 4. Find Infomap communities containing fraud nodes
# ---------------------------------------------------------

fraud_communities = set(
    infomap_df.loc[
        infomap_df["node_id"].isin(fraud_nodes),
        "infomap_id"
    ]
)

nodes_in_fraud_communities = set(
    infomap_df.loc[
        infomap_df["infomap_id"].isin(fraud_communities),
        "node_id"
    ]
)

print("Fraud-related Infomap communities:", len(fraud_communities))
print("Nodes in fraud-related Infomap communities:", len(nodes_in_fraud_communities))

Fraud-related Infomap communities: 2586
Nodes in fraud-related Infomap communities: 1125121


### 7.5 Candidate normal nodes from the rest of the graph

Constructs an additional pool of normal candidates outside the fraud communities.


In [ ]:
# ---------------------------------------------------------
# 5. Find direct neighbors of fraudulent nodes
# ---------------------------------------------------------

fraud_out_edges = edges[edges["src"].isin(fraud_nodes)]
fraud_in_edges = edges[edges["dst"].isin(fraud_nodes)]

direct_neighbors_of_fraud = set(fraud_out_edges["dst"]).union(
    set(fraud_in_edges["src"])
)

print("Direct neighbors of fraud nodes:", len(direct_neighbors_of_fraud))

Direct neighbors of fraud nodes: 1113021


### 7.6 Sample community-aware negative labels

Samples normal nodes from the community-aware and random pools according to the intended negative-label strategy.


In [ ]:
# ---------------------------------------------------------
# 6. Select 25,000 normal nodes from fraud-related Infomap communities
# ---------------------------------------------------------

community_normal_candidates = (
    nodes_in_fraud_communities
    - fraud_nodes
    - direct_neighbors_of_fraud
)

community_normal_candidates = list(community_normal_candidates)

print("Community normal candidates:", len(community_normal_candidates))

if len(community_normal_candidates) < TARGET_COMMUNITY_NORMALS:
    raise ValueError(
        f"Only {len(community_normal_candidates)} community normal candidates found. "
        f"Need {TARGET_COMMUNITY_NORMALS}."
    )

rng = np.random.default_rng(RANDOM_SEED)

community_normal_nodes = set(
    rng.choice(
        community_normal_candidates,
        size=TARGET_COMMUNITY_NORMALS,
        replace=False
    )
)

print("Selected community normals:", len(community_normal_nodes))

Community normal candidates: 198271
Selected community normals: 25000


### 7.7 Build the final label table

Combines fraud labels and sampled normal labels into one semi-supervised node-label table.


In [ ]:
# ---------------------------------------------------------
# 7. Select 20,000 random normal nodes from the rest of the graph
# ---------------------------------------------------------

all_graph_nodes = set(node_mapping["node_id"].astype(int))

random_normal_candidates = (
    all_graph_nodes
    - fraud_nodes
    - direct_neighbors_of_fraud
    - community_normal_nodes
    - nodes_in_fraud_communities
)

random_normal_candidates = list(random_normal_candidates)

print("Random normal candidates from rest of graph:", len(random_normal_candidates))

if len(random_normal_candidates) < TARGET_RANDOM_NORMALS:
    raise ValueError(
        f"Only {len(random_normal_candidates)} random normal candidates found. "
        f"Need {TARGET_RANDOM_NORMALS}."
    )

random_normal_nodes = set(
    rng.choice(
        random_normal_candidates,
        size=TARGET_RANDOM_NORMALS,
        replace=False
    )
)

print("Selected random normals:", len(random_normal_nodes))

Random normal candidates from rest of graph: 516021
Selected random normals: 20000


### 7.8 Label sanity checks

Checks class counts, duplicate node IDs, and overlap between fraud and normal labels.


In [ ]:
# ---------------------------------------------------------
# 8. Create final GNN label table
# ---------------------------------------------------------

fraud_labels = pd.DataFrame({
    "node_id": list(fraud_nodes),
    "label": 1,
    "label_type": "fraud_merged"
})

community_normal_labels = pd.DataFrame({
    "node_id": list(community_normal_nodes),
    "label": 0,
    "label_type": "normal_fraud_infomap_community_not_direct_neighbor"
})

random_normal_labels = pd.DataFrame({
    "node_id": list(random_normal_nodes),
    "label": 0,
    "label_type": "normal_random_rest_of_graph"
})

gnn_labels = pd.concat(
    [fraud_labels, community_normal_labels, random_normal_labels],
    ignore_index=True
)

gnn_labels = gnn_labels.merge(
    node_mapping[["node_id", "address"]],
    on="node_id",
    how="left"
)

gnn_labels = gnn_labels[["node_id", "address", "label", "label_type"]]

print(gnn_labels["label"].value_counts())
print(gnn_labels["label_type"].value_counts())
print("Total labelled nodes:", len(gnn_labels))

label
0    45000
1     4466
Name: count, dtype: int64
label_type
normal_fraud_infomap_community_not_direct_neighbor    25000
normal_random_rest_of_graph                           20000
fraud_merged                                           4466
Name: count, dtype: int64
Total labelled nodes: 49466


### 7.9 Save baseline labels

Exports the constructed label table for GraphSAGE training.


In [ ]:
# ---------------------------------------------------------
# 9. Save final label file
# ---------------------------------------------------------

LABEL_OUTPUT_PATH = BASELINE_LABELS_PATH

gnn_labels.to_csv(LABEL_OUTPUT_PATH, index=False)

print("Saved:", LABEL_OUTPUT_PATH)

Saved: /content/drive/MyDrive/Graph_Mining_Project/outputs/03_labels/gnn_labels_baseline_sampled_normals.csv


## 8. Baseline GraphSAGE training setup

Defines paths for the basic-feature graph, labels, and node mapping used in the first GraphSAGE experiment.


In [ ]:

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------
GRAPH_PATH = BASIC_GRAPH_PATH
LABEL_PATH = BASELINE_LABELS_PATH
NODE_MAPPING_PATH = NODE_MAPPING_FILE_PATH


### 8.1 Load graph and labels

Loads the PyTorch Geometric graph, the node label table, and the address mapping.


In [ ]:

# ---------------------------------------------------------
# 2. Load graph and labels
# ---------------------------------------------------------

data = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

labels_df = pd.read_csv(LABEL_PATH)
node_mapping = pd.read_csv(NODE_MAPPING_PATH)

labels_df["node_id"] = labels_df["node_id"].astype(int)
labels_df["label"] = labels_df["label"].astype(int)

print(data)
print(labels_df["label"].value_counts())
print("Total labelled nodes:", len(labels_df))


Data(x=[1828910, 20], edge_index=[2, 6092594], edge_attr=[6092594, 2], num_nodes=1828910)
label
0    45000
1     4466
Name: count, dtype: int64
Total labelled nodes: 49466


### 8.2 Construct train/validation/test masks

Splits only the labeled nodes into train, validation, and test masks while the GraphSAGE encoder still receives the full graph structure.


In [ ]:

# ---------------------------------------------------------
# 3. Create y vector
# ---------------------------------------------------------
# -1 = unlabeled
#  0 = normal
#  1 = fraud

num_nodes = data.num_nodes

y = torch.full(
    (num_nodes,),
    -1,
    dtype=torch.long
)

labelled_node_ids = labels_df["node_id"].values
label_values = labels_df["label"].values

y[labelled_node_ids] = torch.tensor(
    label_values,
    dtype=torch.long
)

data.y = y

print("Unlabelled nodes:", (data.y == -1).sum().item())
print("Normal labelled nodes:", (data.y == 0).sum().item())
print("Fraud labelled nodes:", (data.y == 1).sum().item())


Unlabelled nodes: 1779444
Normal labelled nodes: 45000
Fraud labelled nodes: 4466


### 8.3 Define the GraphSAGE model

Implements the baseline GraphSAGE node classifier using the graph topology and engineered node features.


In [ ]:

# ---------------------------------------------------------
# 4. Train / validation / test split
# ---------------------------------------------------------

train_ids, temp_ids = train_test_split(
    labelled_node_ids,
    test_size=0.30,
    random_state=42,
    stratify=label_values
)

temp_labels = y[temp_ids].numpy()

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_ids] = True
val_mask[val_ids] = True
test_mask[test_ids] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print("Train nodes:", data.train_mask.sum().item())
print("Validation nodes:", data.val_mask.sum().item())
print("Test nodes:", data.test_mask.sum().item())


Train nodes: 34626
Validation nodes: 7420
Test nodes: 7420


### 8.4 Prepare device and class weighting

Moves data to GPU when available and computes class weights to compensate for fraud/normal imbalance during training.


In [ ]:
# Use the reusable one-hidden-layer GraphSAGE architecture defined in the shared utilities section.
GraphSAGE = GraphSAGEOneHidden


### 8.5 Training and evaluation functions

Defines the training step and evaluation metrics used for model selection and reporting.


In [ ]:

# ---------------------------------------------------------
# 6. Device, model, optimizer
# ---------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

data = data.to(device)

model = GraphSAGE(
    in_channels=data.x.shape[1],
    hidden_channels=64,
    out_channels=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)


Using device: cuda


### 8.6 Train the baseline model

Trains the GraphSAGE classifier and monitors validation performance.


In [ ]:

# ---------------------------------------------------------
# 7. Class weights for imbalance
# ---------------------------------------------------------

train_labels = data.y[data.train_mask]

num_normal = (train_labels == 0).sum().item()
num_fraud = (train_labels == 1).sum().item()

class_weights = torch.tensor(
    [
        1.0 / num_normal,
        1.0 / num_fraud
    ],
    dtype=torch.float,
    device=device
)

class_weights = class_weights / class_weights.sum() * 2

print("Class weights:", class_weights)


Class weights: tensor([0.1806, 1.8194], device='cuda:0')


### 8.7 Final test evaluation

Evaluates the selected model on the held-out test labels.


In [ ]:
# The standard train_one_epoch() and evaluate() functions are defined once in the shared utilities section.


### 8.8 Save the trained baseline model

Persists the best-performing GraphSAGE model checkpoint.


In [ ]:

# ---------------------------------------------------------
# 9. Train model
# ---------------------------------------------------------

EPOCHS = 100

best_val_pr_auc = 0
best_model_path = BEST_GRAPHSAGE_BASIC_1H_PATH

for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch()

    if epoch % 10 == 0:
        train_acc, train_roc, train_pr, _, _, _ = evaluate(data.train_mask)
        val_acc, val_roc, val_pr, _, _, _ = evaluate(data.val_mask)

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Train PR-AUC: {train_pr:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val ROC-AUC: {val_roc:.4f} | "
            f"Val PR-AUC: {val_pr:.4f}"
        )

        if val_pr > best_val_pr_auc:
            best_val_pr_auc = val_pr

            torch.save(
                model.state_dict(),
                best_model_path
            )

            print("Saved best model.")


Epoch 010 | Loss: 0.5919 | Train Acc: 0.3795 | Train PR-AUC: 0.5326 | Val Acc: 0.3717 | Val ROC-AUC: 0.9088 | Val PR-AUC: 0.5234
Saved best model.
Epoch 020 | Loss: 0.4797 | Train Acc: 0.8059 | Train PR-AUC: 0.7318 | Val Acc: 0.7980 | Val ROC-AUC: 0.9489 | Val PR-AUC: 0.7140
Saved best model.
Epoch 030 | Loss: 0.4018 | Train Acc: 0.8476 | Train PR-AUC: 0.8073 | Val Acc: 0.8384 | Val ROC-AUC: 0.9659 | Val PR-AUC: 0.7906
Saved best model.
Epoch 040 | Loss: 0.3502 | Train Acc: 0.8689 | Train PR-AUC: 0.8484 | Val Acc: 0.8623 | Val ROC-AUC: 0.9747 | Val PR-AUC: 0.8324
Saved best model.
Epoch 050 | Loss: 0.3048 | Train Acc: 0.9091 | Train PR-AUC: 0.8735 | Val Acc: 0.9054 | Val ROC-AUC: 0.9789 | Val PR-AUC: 0.8581
Saved best model.
Epoch 060 | Loss: 0.2652 | Train Acc: 0.9206 | Train PR-AUC: 0.8929 | Val Acc: 0.9193 | Val ROC-AUC: 0.9815 | Val PR-AUC: 0.8788
Saved best model.
Epoch 070 | Loss: 0.2378 | Train Acc: 0.9242 | Train PR-AUC: 0.9060 | Val Acc: 0.9228 | Val ROC-AUC: 0.9830 | Val PR-A

### 8.9 Inference over all graph nodes

Uses the trained classifier to compute fraud probabilities for every node in the graph.


In [ ]:

# ---------------------------------------------------------
# 10. Final test evaluation
# ---------------------------------------------------------

model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

test_acc, test_roc, test_pr, test_preds, test_probs, test_labels = evaluate(
    data.test_mask
)

print("\nFinal test results")
print("Test accuracy:", test_acc)
print("Test ROC-AUC:", test_roc)
print("Test PR-AUC:", test_pr)

print("\nClassification report:")
print(
    classification_report(
        test_labels,
        test_preds,
        target_names=["normal", "fraud"]
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(test_labels, test_preds))





Final test results
Test accuracy: 0.9388140439987183
Test ROC-AUC: 0.9893173023770039
Test PR-AUC: 0.9075922177418283

Classification report:
              precision    recall  f1-score   support

      normal       1.00      0.93      0.97      6750
       fraud       0.60      0.98      0.74       670

    accuracy                           0.94      7420
   macro avg       0.80      0.96      0.85      7420
weighted avg       0.96      0.94      0.95      7420


Confusion matrix:
[[6309  441]
 [  13  657]]


### Result comment: baseline GraphSAGE on engineered node features

The default-argmax baseline GraphSAGE now shows very strong ranking performance on the held-out test set, with ROC-AUC ≈ 0.989 and PR-AUC ≈ 0.908. Fraud recall remains extremely high, around 0.98, meaning that the model recovers almost all labelled fraud nodes. However, fraud precision is only about 0.60, with 441 normal nodes incorrectly classified as fraud. This confirms that the default argmax rule is still too recall-oriented: the model is good at ranking suspicious nodes, but its raw decision threshold produces too many false positives for practical use. Threshold tuning is therefore necessary.


### 8.10 Save all-node predictions

Exports node-level fraud probabilities and predicted labels for downstream inspection.


In [ ]:
# ---------------------------------------------------------
# 11. Predict fraud probability for every node
# ---------------------------------------------------------
model.eval()

with torch.no_grad():
    out = model(data.x, data.edge_index)
    probs = F.softmax(out, dim=1)
    fraud_probs = probs[:, 1].detach().cpu().numpy()

pred_df = pd.DataFrame({
    "node_id": np.arange(data.num_nodes),
    "fraud_probability": fraud_probs
})

node_mapping["node_id"] = node_mapping["node_id"].astype(int)

pred_df = pred_df.merge(
    node_mapping,
    on="node_id",
    how="left"
)

# Mark whether the node was labelled or unlabeled
y_cpu = data.y.detach().cpu().numpy()

pred_df["known_label"] = y_cpu
pred_df["is_labelled"] = pred_df["known_label"] != -1

pred_df = pred_df.sort_values(
    "fraud_probability",
    ascending=False
)

PRED_OUTPUT_PATH = GRAPHSAGE_BASIC_PREDICTIONS_PATH

pred_df.to_csv(
    PRED_OUTPUT_PATH,
    index=False
)

print("\nSaved predictions:")
print(PRED_OUTPUT_PATH)

display(pred_df.head(20))


Saved predictions:
/content/drive/MyDrive/Graph_Mining_Project/outputs/06_predictions/graphsage_basic_features_predictions.csv


,node_id,fraud_probability,address,known_label,is_labelled
137667,137667,0.999954,0xfdeced66af5596e1a4567b52a71060e3ba470eb3,1,True
4478,4478,0.999862,0xad6eaa735d9df3d7696fd03984379dae02ed8862,1,True
29562,29562,0.999860,0xcffad3200574698b78f32232aa9d63eabd290703,1,True
4260,4260,0.999858,0x53ffd4cc068dad23e1016daaa2706679fe884ca6,1,True
5605,5605,0.999853,0xb5d85cbf7cb3ee0d56b3bb207d5fc4b82f43f511,1,True
42941,42941,0.999850,0x0003b5aa5e30e97fcc596bb5d0f3a75255e08d4e,1,True
1573,1573,0.999847,0x017a71c00d41caf9088a718093874bb069436a79,1,True
217035,217035,0.999832,0x89214882e9a2764fc0ad1325fbf8e9cbe11e1063,1,True
276,276,0.999783,0x091d1c972cb1648537a2ba78eaba371b1ce18336,1,True
1241192,1241192,0.999778,0xd0a3f823b16450482e34f1217ddb24817c60cc0c,1,True


## 9. Threshold-tuned GraphSAGE experiment

Repeats GraphSAGE training with validation-based threshold selection so that classification can be tuned beyond the default probability cutoff (0.5).


In [ ]:
# =========================================================
# GraphSAGE semi-supervised node classification
# with validation-selected fraud threshold
# =========================================================

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------

GRAPH_PATH = BASIC_GRAPH_PATH
LABEL_PATH = BASELINE_LABELS_PATH
NODE_MAPPING_PATH = NODE_MAPPING_FILE_PATH

# ---------------------------------------------------------
# 2. Load graph and labels
# ---------------------------------------------------------

data = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

labels_df = pd.read_csv(LABEL_PATH)
node_mapping = pd.read_csv(NODE_MAPPING_PATH)

labels_df["node_id"] = labels_df["node_id"].astype(int)
labels_df["label"] = labels_df["label"].astype(int)

print(data)
print(labels_df["label"].value_counts())
print("Total labelled nodes:", len(labels_df))

# ---------------------------------------------------------
# 3. Create y vector
# ---------------------------------------------------------
# -1 = unlabeled
#  0 = normal
#  1 = fraud

num_nodes = data.num_nodes

y = torch.full(
    (num_nodes,),
    -1,
    dtype=torch.long
)

labelled_node_ids = labels_df["node_id"].values
label_values = labels_df["label"].values

y[labelled_node_ids] = torch.tensor(
    label_values,
    dtype=torch.long
)

data.y = y

print("Unlabelled nodes:", (data.y == -1).sum().item())
print("Normal labelled nodes:", (data.y == 0).sum().item())
print("Fraud labelled nodes:", (data.y == 1).sum().item())

# ---------------------------------------------------------
# 4. Train / validation / test split
# ---------------------------------------------------------

train_ids, temp_ids = train_test_split(
    labelled_node_ids,
    test_size=0.30,
    random_state=42,
    stratify=label_values
)

temp_labels = y[temp_ids].numpy()

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_ids] = True
val_mask[val_ids] = True
test_mask[test_ids] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print("Train nodes:", data.train_mask.sum().item())
print("Validation nodes:", data.val_mask.sum().item())
print("Test nodes:", data.test_mask.sum().item())

# ---------------------------------------------------------
# Select reusable GraphSAGE architecture
# ---------------------------------------------------------

GraphSAGE = GraphSAGEOneHidden

# ---------------------------------------------------------
# 6. Device, model, optimizer
# ---------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

data = data.to(device)

model = GraphSAGE(
    in_channels=data.x.shape[1],
    hidden_channels=64,
    out_channels=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

# ---------------------------------------------------------
# 7. Class weights for imbalance
# ---------------------------------------------------------

train_labels = data.y[data.train_mask]

num_normal = (train_labels == 0).sum().item()
num_fraud = (train_labels == 1).sum().item()

class_weights = torch.tensor(
    [
        1.0 / num_normal,
        1.0 / num_fraud
    ],
    dtype=torch.float,
    device=device
)

class_weights = class_weights / class_weights.sum() * 2

print("Class weights:", class_weights)

# ---------------------------------------------------------
# Reuse standard training and evaluation functions
# ---------------------------------------------------------

# train_one_epoch() and evaluate() are defined in the shared utilities section.

# ---------------------------------------------------------
# 9. Train model
# ---------------------------------------------------------

EPOCHS = 100

best_val_pr_auc = 0
best_model_path = BEST_GRAPHSAGE_BASIC_THRESHOLD_TUNED_PATH

for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch()

    if epoch % 10 == 0:
        train_acc, train_roc, train_pr, _, _, _ = evaluate(data.train_mask)
        val_acc, val_roc, val_pr, _, _, _ = evaluate(data.val_mask)

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Train PR-AUC: {train_pr:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val ROC-AUC: {val_roc:.4f} | "
            f"Val PR-AUC: {val_pr:.4f}"
        )

        if val_pr > best_val_pr_auc:
            best_val_pr_auc = val_pr

            torch.save(
                model.state_dict(),
                best_model_path
            )

            print("Saved best model.")

# ---------------------------------------------------------
# 10. Load best model
# ---------------------------------------------------------

model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

# ---------------------------------------------------------
# 11. Select best fraud threshold on validation set
# ---------------------------------------------------------
# Choose the threshold that maximizes F1-score on validation data.

val_acc, val_roc, val_pr, val_preds_argmax, val_probs, val_labels = evaluate(
    data.val_mask
)

thresholds = np.arange(0.05, 0.96, 0.01)

threshold_results = []

for threshold in thresholds:
    val_preds_thresholded = (val_probs >= threshold).astype(int)

    precision = precision_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    recall = recall_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    f1 = f1_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results = pd.DataFrame(threshold_results)

best_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold = float(best_row["threshold"])

print("\nBest threshold selected on validation set:")
print(best_row)

# Save threshold search results
threshold_results_path = GRAPHSAGE_BASIC_THRESHOLD_TABLE_PATH

threshold_results.to_csv(
    threshold_results_path,
    index=False
)

print("Saved threshold results:", threshold_results_path)

# ---------------------------------------------------------
# 12. Final test evaluation with selected threshold
# ---------------------------------------------------------

test_acc_argmax, test_roc, test_pr, test_preds_argmax, test_probs, test_labels = evaluate(
    data.test_mask
)

test_preds_thresholded = (test_probs >= best_threshold).astype(int)

print("\nFinal test results with validation-selected threshold")
print("Selected threshold:", best_threshold)
print("Test ROC-AUC:", test_roc)
print("Test PR-AUC:", test_pr)

print("\nClassification report:")
print(
    classification_report(
        test_labels,
        test_preds_thresholded,
        target_names=["normal", "fraud"],
        zero_division=0
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(test_labels, test_preds_thresholded))

# ---------------------------------------------------------
# 13. Predict fraud probability for every node
# ---------------------------------------------------------

model.eval()

with torch.no_grad():
    out = model(data.x, data.edge_index)
    probs = F.softmax(out, dim=1)
    fraud_probs = probs[:, 1].detach().cpu().numpy()

pred_df = pd.DataFrame({
    "node_id": np.arange(data.num_nodes),
    "fraud_probability": fraud_probs
})

node_mapping["node_id"] = node_mapping["node_id"].astype(int)

pred_df = pred_df.merge(
    node_mapping,
    on="node_id",
    how="left"
)

# Mark whether the node was labelled or unlabeled
y_cpu = data.y.detach().cpu().numpy()

pred_df["known_label"] = y_cpu
pred_df["is_labelled"] = pred_df["known_label"] != -1

# Add thresholded prediction
pred_df["predicted_label_thresholded"] = (
    pred_df["fraud_probability"] >= best_threshold
).astype(int)

pred_df["predicted_label_name"] = pred_df[
    "predicted_label_thresholded"
].map({
    0: "normal",
    1: "fraud"
})

pred_df = pred_df.sort_values(
    "fraud_probability",
    ascending=False
)

PRED_OUTPUT_PATH = GRAPHSAGE_BASIC_THRESHOLD_TUNED_PREDICTIONS_PATH

pred_df.to_csv(
    PRED_OUTPUT_PATH,
    index=False
)

print("\nSaved predictions:")
print(PRED_OUTPUT_PATH)

display(pred_df.head(20))

Data(x=[1828910, 20], edge_index=[2, 6092594], edge_attr=[6092594, 2], num_nodes=1828910)
label
0    45000
1     4466
Name: count, dtype: int64
Total labelled nodes: 49466
Unlabelled nodes: 1779444
Normal labelled nodes: 45000
Fraud labelled nodes: 4466
Train nodes: 34626
Validation nodes: 7420
Test nodes: 7420
Using device: cuda
Class weights: tensor([0.1806, 1.8194], device='cuda:0')
Epoch 010 | Loss: 0.5997 | Train Acc: 0.6601 | Train PR-AUC: 0.4169 | Val Acc: 0.6567 | Val ROC-AUC: 0.8972 | Val PR-AUC: 0.4216
Saved best model.
Epoch 020 | Loss: 0.4713 | Train Acc: 0.8256 | Train PR-AUC: 0.6036 | Val Acc: 0.8168 | Val ROC-AUC: 0.9438 | Val PR-AUC: 0.6223
Saved best model.
Epoch 030 | Loss: 0.3846 | Train Acc: 0.8423 | Train PR-AUC: 0.7729 | Val Acc: 0.8353 | Val ROC-AUC: 0.9671 | Val PR-AUC: 0.7787
Saved best model.
Epoch 040 | Loss: 0.3278 | Train Acc: 0.8823 | Train PR-AUC: 0.8397 | Val Acc: 0.8794 | Val ROC-AUC: 0.9753 | Val PR-AUC: 0.8398
Saved best model.
Epoch 050 | Loss: 0.283

,node_id,fraud_probability,address,known_label,is_labelled,predicted_label_thresholded,predicted_label_name
137667,137667,0.999486,0xfdeced66af5596e1a4567b52a71060e3ba470eb3,1,True,1,fraud
4478,4478,0.999252,0xad6eaa735d9df3d7696fd03984379dae02ed8862,1,True,1,fraud
169853,169853,0.999224,0x113a1c1294deb0cae51b17a3fdd4c2cf7ba935df,1,True,1,fraud
5605,5605,0.999221,0xb5d85cbf7cb3ee0d56b3bb207d5fc4b82f43f511,1,True,1,fraud
4260,4260,0.999197,0x53ffd4cc068dad23e1016daaa2706679fe884ca6,1,True,1,fraud
1241192,1241192,0.999191,0xd0a3f823b16450482e34f1217ddb24817c60cc0c,1,True,1,fraud
1573,1573,0.999180,0x017a71c00d41caf9088a718093874bb069436a79,1,True,1,fraud
29562,29562,0.999144,0xcffad3200574698b78f32232aa9d63eabd290703,1,True,1,fraud
64141,64141,0.999107,0x50de1986cf5befe0c8ee74d6df6dffe842dc2632,1,True,1,fraud
1625562,1625562,0.999085,0xa9d1e08c7793af67e9d92fe308d5697fb81d3e43,1,True,1,fraud


### Result comment: validation-threshold GraphSAGE on baseline labels

The validation-selected threshold gives the strongest practical baseline. Compared with the default argmax version, fraud precision increases from about 0.60 to about 0.82, while fraud recall remains high at about 0.92. False positives decrease from 441 to 132, while false negatives increase from 13 to 51. This is a much better operational trade-off: the model still detects most fraud nodes, but it greatly reduces unnecessary fraud alerts. This run reaches ROC-AUC ≈ 0.989 and PR-AUC ≈ 0.911, making it the best overall practical model in terms of precision-recall balance. This high precision however is probably due to the labelled set structure as we will show and tackle in the next section.



## 10. Alternative community-aware label set

Builds a second label table using fraud nodes and sampled normal nodes designed to be more structurally related to fraud communities while avoiding direct fraud neighbors and also more high activity.


In [ ]:
# Fraud nodes:
#   - all nodes in fraud_addresses_merged_internal_and_external.csv
#
# Normal nodes:
#   1. 15,000 nodes from the same Infomap communities as fraud nodes,
#      excluding direct neighbors of fraud nodes
#   2. 10,000 high-activity nodes
#   3. 20,000 random nodes from the rest of the graph with out_degree > 1
#
# Output:
#   gnn_labels_baseline_sampled_normals.csv


# ---------------------------------------------------------
# 1. Paths and parameters
# ---------------------------------------------------------

NODE_MAPPING_PATH = NODE_MAPPING_FILE_PATH
NODE_FEATURES_PATH = NODE_BASIC_FEATURES_PATH
AGG_EDGES_PATH = AGG_DIRECTED_EDGES_PATH
FRAUD_PATH = MERGED_FRAUD_ADDRESSES_PATH

INFOMAP_PATH = INFOMAP_COMMUNITIES_PATH

TARGET_COMMUNITY_NORMALS = 15000
TARGET_HIGH_ACTIVITY_NORMALS = 10000
TARGET_RANDOM_NORMALS = 20000

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# ---------------------------------------------------------
# 2. Load files
# ---------------------------------------------------------

node_mapping = pd.read_csv(NODE_MAPPING_PATH)
node_features = pd.read_csv(NODE_FEATURES_PATH)
edges = pd.read_csv(AGG_EDGES_PATH)
fraud_df = pd.read_csv(FRAUD_PATH)
infomap_df = pd.read_csv(INFOMAP_PATH)

# ---------------------------------------------------------
# 3. Standardize columns
# ---------------------------------------------------------

node_mapping["address"] = (
    node_mapping["address"]
    .astype(str)
    .str.lower()
    .str.strip()
)

node_mapping["node_id"] = node_mapping["node_id"].astype(int)

edges["src"] = edges["src"].astype(int)
edges["dst"] = edges["dst"].astype(int)

if "address" in fraud_df.columns:
    fraud_df["address"] = (
        fraud_df["address"]
        .astype(str)
        .str.lower()
        .str.strip()
    )

if "address" in infomap_df.columns:
    infomap_df["address"] = (
        infomap_df["address"]
        .astype(str)
        .str.lower()
        .str.strip()
    )

# Ensure node_features has node_id
if "node_id" not in node_features.columns:
    node_features["node_id"] = node_features.index

node_features["node_id"] = node_features["node_id"].astype(int)

print("Nodes in graph:", len(node_mapping))
print("Aggregated edges:", len(edges))
print("Node features:", node_features.shape)
print("Merged fraud accounts:", len(fraud_df))
print("Infomap rows:", len(infomap_df))

# ---------------------------------------------------------
# 4. Get fraud node IDs
# ---------------------------------------------------------

if "node_id" in fraud_df.columns:
    fraud_df["node_id"] = pd.to_numeric(
        fraud_df["node_id"],
        errors="coerce"
    )
    fraud_df = fraud_df.dropna(subset=["node_id"])
    fraud_df["node_id"] = fraud_df["node_id"].astype(int)

else:
    fraud_df = fraud_df.merge(
        node_mapping[["address", "node_id"]],
        on="address",
        how="inner"
    )

fraud_nodes = set(fraud_df["node_id"].astype(int))

print("Fraud nodes found in graph:", len(fraud_nodes))

# ---------------------------------------------------------
# 5. Attach node_id to Infomap table
# ---------------------------------------------------------

if "node_id" not in infomap_df.columns:
    infomap_df = infomap_df.merge(
        node_mapping[["address", "node_id"]],
        on="address",
        how="inner"
    )

infomap_df["node_id"] = infomap_df["node_id"].astype(int)

print("Infomap nodes found in graph:", len(infomap_df))

# ---------------------------------------------------------
# 6. Compute direct neighbors of fraud nodes
# ---------------------------------------------------------

fraud_out_edges = edges[edges["src"].isin(fraud_nodes)]
fraud_in_edges = edges[edges["dst"].isin(fraud_nodes)]

direct_neighbors_of_fraud = set(fraud_out_edges["dst"]).union(
    set(fraud_in_edges["src"])
)

print("Direct neighbors of fraud nodes:", len(direct_neighbors_of_fraud))

# ---------------------------------------------------------
# 7. Find Infomap communities containing fraud nodes
# ---------------------------------------------------------

fraud_communities = set(
    infomap_df.loc[
        infomap_df["node_id"].isin(fraud_nodes),
        "infomap_id"
    ]
)

nodes_in_fraud_communities = set(
    infomap_df.loc[
        infomap_df["infomap_id"].isin(fraud_communities),
        "node_id"
    ]
)

print("Fraud-related Infomap communities:", len(fraud_communities))
print("Nodes in fraud-related communities:", len(nodes_in_fraud_communities))

# ---------------------------------------------------------
# 8. Sample 15,000 community-based normal nodes
# ---------------------------------------------------------

community_normal_candidates = (
    nodes_in_fraud_communities
    - fraud_nodes
    - direct_neighbors_of_fraud
)

community_normal_candidates = np.array(
    list(community_normal_candidates),
    dtype=int
)

print("Community normal candidates:", len(community_normal_candidates))

if len(community_normal_candidates) < TARGET_COMMUNITY_NORMALS:
    raise ValueError(
        f"Only {len(community_normal_candidates)} community normal candidates found. "
        f"Need {TARGET_COMMUNITY_NORMALS}."
    )

community_normal_nodes = set(
    rng.choice(
        community_normal_candidates,
        size=TARGET_COMMUNITY_NORMALS,
        replace=False
    )
)

print("Selected community normals:", len(community_normal_nodes))

# ---------------------------------------------------------
# 9. Sample 10,000 high-activity normal nodes
# ---------------------------------------------------------
# High activity is defined using total_tx if available.
# If total_tx is not available, it is approximated with in_degree + out_degree.

features = node_features.copy()

if "total_tx" not in features.columns:
    if {"in_degree", "out_degree"}.issubset(features.columns):
        features["total_tx"] = features["in_degree"] + features["out_degree"]
    else:
        raise ValueError(
            "node_features must contain either 'total_tx' or both 'in_degree' and 'out_degree'."
        )

all_graph_nodes = set(node_mapping["node_id"].astype(int))

excluded_before_high_activity = (
    fraud_nodes
    | direct_neighbors_of_fraud
    | community_normal_nodes
)

high_activity_candidates = features[
    ~features["node_id"].isin(excluded_before_high_activity)
].copy()

# Keep only nodes that exist in the graph
high_activity_candidates = high_activity_candidates[
    high_activity_candidates["node_id"].isin(all_graph_nodes)
]

# Sort by activity descending
high_activity_candidates = high_activity_candidates.sort_values(
    "total_tx",
    ascending=False
)

print("High-activity candidates:", len(high_activity_candidates))

if len(high_activity_candidates) < TARGET_HIGH_ACTIVITY_NORMALS:
    raise ValueError(
        f"Only {len(high_activity_candidates)} high-activity candidates found. "
        f"Need {TARGET_HIGH_ACTIVITY_NORMALS}."
    )

high_activity_normal_nodes = set(
    high_activity_candidates
    .head(TARGET_HIGH_ACTIVITY_NORMALS)["node_id"]
    .astype(int)
)

print("Selected high-activity normals:", len(high_activity_normal_nodes))

# ---------------------------------------------------------
# 10. Sample 20,000 random normal nodes with out_degree > 1
# ---------------------------------------------------------

excluded_before_random = (
    fraud_nodes
    | direct_neighbors_of_fraud
    | community_normal_nodes
    | high_activity_normal_nodes
)

if "out_degree" not in features.columns:
    # Compute unique-neighbor out_degree from ethereum_aggregated_directed_edges.csv
    out_degree_df = (
        edges.groupby("src")["dst"]
        .nunique()
        .reset_index()
        .rename(columns={"src": "node_id", "dst": "out_degree"})
    )

    features = features.drop(columns=["out_degree"], errors="ignore")
    features = features.merge(
        out_degree_df,
        on="node_id",
        how="left"
    )

    features["out_degree"] = features["out_degree"].fillna(0)

random_candidates = features[
    (features["out_degree"] > 1)
    & (~features["node_id"].isin(excluded_before_random))
].copy()

random_candidates = random_candidates[
    random_candidates["node_id"].isin(all_graph_nodes)
]

random_candidate_nodes = np.array(
    random_candidates["node_id"].astype(int).unique(),
    dtype=int
)

print("Random candidates with out_degree > 1:", len(random_candidate_nodes))

if len(random_candidate_nodes) < TARGET_RANDOM_NORMALS:
    raise ValueError(
        f"Only {len(random_candidate_nodes)} random candidates with out_degree > 1 found. "
        f"Need {TARGET_RANDOM_NORMALS}."
    )

random_normal_nodes = set(
    rng.choice(
        random_candidate_nodes,
        size=TARGET_RANDOM_NORMALS,
        replace=False
    )
)

print("Selected random normals:", len(random_normal_nodes))

# ---------------------------------------------------------
# 11. Create final label table
# ---------------------------------------------------------

fraud_labels = pd.DataFrame({
    "node_id": list(fraud_nodes),
    "label": 1,
    "label_type": "fraud_merged"
})

community_normal_labels = pd.DataFrame({
    "node_id": list(community_normal_nodes),
    "label": 0,
    "label_type": "normal_same_fraud_infomap_community_not_direct_neighbor"
})

high_activity_normal_labels = pd.DataFrame({
    "node_id": list(high_activity_normal_nodes),
    "label": 0,
    "label_type": "normal_high_activity"
})

random_normal_labels = pd.DataFrame({
    "node_id": list(random_normal_nodes),
    "label": 0,
    "label_type": "normal_random_out_degree_gt_1"
})

gnn_labels = pd.concat(
    [
        fraud_labels,
        community_normal_labels,
        high_activity_normal_labels,
        random_normal_labels
    ],
    ignore_index=True
)

# Safety: remove accidental duplicate node_ids.
# If a duplicate exists, fraud label dominates.
gnn_labels = gnn_labels.sort_values(
    "label",
    ascending=False
).drop_duplicates(
    subset=["node_id"],
    keep="first"
)

gnn_labels = gnn_labels.merge(
    node_mapping[["node_id", "address"]],
    on="node_id",
    how="left"
)

gnn_labels = gnn_labels[
    ["node_id", "address", "label", "label_type"]
]

print("\nFinal label counts:")
print(gnn_labels["label"].value_counts())

print("\nLabel type counts:")
print(gnn_labels["label_type"].value_counts())

print("\nTotal labelled nodes:", len(gnn_labels))

# ---------------------------------------------------------
# 12. Save final labels
# ---------------------------------------------------------

LABEL_OUTPUT_PATH = COMMUNITY_AWARE_LABELS_PATH

gnn_labels.to_csv(
    LABEL_OUTPUT_PATH,
    index=False
)

print("\nSaved:", LABEL_OUTPUT_PATH)

Nodes in graph: 1828910
Aggregated edges: 2948955
Node features: (1828910, 21)
Merged fraud accounts: 4466
Infomap rows: 1686534
Fraud nodes found in graph: 4466
Infomap nodes found in graph: 1686380
Direct neighbors of fraud nodes: 1113021
Fraud-related Infomap communities: 2586
Nodes in fraud-related communities: 1125121
Community normal candidates: 198271
Selected community normals: 15000
High-activity candidates: 699292
Selected high-activity normals: 10000
Random candidates with out_degree > 1: 93092
Selected random normals: 20000

Final label counts:
label
0    45000
1     4466
Name: count, dtype: int64

Label type counts:
label_type
normal_random_out_degree_gt_1                              20000
normal_same_fraud_infomap_community_not_direct_neighbor    15000
normal_high_activity                                       10000
fraud_merged                                                4466
Name: count, dtype: int64

Total labelled nodes: 49466

Saved: /content/drive/MyDrive/Graph_

## 11. GraphSAGE on the alternative label set

Trains and evaluates GraphSAGE using the second label table and saves the corresponding model and predictions.


In [ ]:
# =========================================================
# GraphSAGE semi-supervised node classification
# with validation-selected fraud threshold
# =========================================================

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------

GRAPH_PATH = BASIC_GRAPH_PATH
LABEL_PATH = COMMUNITY_AWARE_LABELS_PATH
NODE_MAPPING_PATH = NODE_MAPPING_FILE_PATH

# ---------------------------------------------------------
# 2. Load graph and labels
# ---------------------------------------------------------

data = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

labels_df = pd.read_csv(LABEL_PATH)
node_mapping = pd.read_csv(NODE_MAPPING_PATH)

labels_df["node_id"] = labels_df["node_id"].astype(int)
labels_df["label"] = labels_df["label"].astype(int)

print(data)
print(labels_df["label"].value_counts())
print("Total labelled nodes:", len(labels_df))

# ---------------------------------------------------------
# 3. Create y vector
# ---------------------------------------------------------
# -1 = unlabeled
#  0 = normal
#  1 = fraud

num_nodes = data.num_nodes

y = torch.full(
    (num_nodes,),
    -1,
    dtype=torch.long
)

labelled_node_ids = labels_df["node_id"].values
label_values = labels_df["label"].values

y[labelled_node_ids] = torch.tensor(
    label_values,
    dtype=torch.long
)

data.y = y

print("Unlabelled nodes:", (data.y == -1).sum().item())
print("Normal labelled nodes:", (data.y == 0).sum().item())
print("Fraud labelled nodes:", (data.y == 1).sum().item())

# ---------------------------------------------------------
# 4. Train / validation / test split
# ---------------------------------------------------------

train_ids, temp_ids = train_test_split(
    labelled_node_ids,
    test_size=0.30,
    random_state=42,
    stratify=label_values
)

temp_labels = y[temp_ids].numpy()

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_ids] = True
val_mask[val_ids] = True
test_mask[test_ids] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print("Train nodes:", data.train_mask.sum().item())
print("Validation nodes:", data.val_mask.sum().item())
print("Test nodes:", data.test_mask.sum().item())

# ---------------------------------------------------------
# Select reusable GraphSAGE architecture
# ---------------------------------------------------------

GraphSAGE = GraphSAGEOneHidden

# ---------------------------------------------------------
# 6. Device, model, optimizer
# ---------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

data = data.to(device)

model = GraphSAGE(
    in_channels=data.x.shape[1],
    hidden_channels=64,
    out_channels=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)


# ---------------------------------------------------------
# 7. Class weights for imbalance
# ---------------------------------------------------------

train_labels = data.y[data.train_mask]

num_normal = (train_labels == 0).sum().item()
num_fraud = (train_labels == 1).sum().item()

class_weights = torch.tensor(
    [
        1.0 / num_normal,
        1.0 / num_fraud
    ],
    dtype=torch.float,
    device=device
)

class_weights = class_weights / class_weights.sum() * 2

print("Class weights:", class_weights)
# ---------------------------------------------------------
# Reuse standard training and evaluation functions
# ---------------------------------------------------------

# train_one_epoch() and evaluate() are defined in the shared utilities section.

# ---------------------------------------------------------
# 9. Train model
# ---------------------------------------------------------

EPOCHS = 150

best_val_pr_auc = 0
best_model_path = BEST_GRAPHSAGE_COMMUNITY_AWARE_PATH

for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch()

    if epoch % 10 == 0:
        train_acc, train_roc, train_pr, _, _, _ = evaluate(data.train_mask)
        val_acc, val_roc, val_pr, _, _, _ = evaluate(data.val_mask)

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Train PR-AUC: {train_pr:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val ROC-AUC: {val_roc:.4f} | "
            f"Val PR-AUC: {val_pr:.4f}"
        )

        if val_pr > best_val_pr_auc:
            best_val_pr_auc = val_pr

            torch.save(
                model.state_dict(),
                best_model_path
            )

            print("Saved best model.")

# ---------------------------------------------------------
# 10. Load best model
# ---------------------------------------------------------

model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

# ---------------------------------------------------------
# 11. Select best fraud threshold on validation set
# ---------------------------------------------------------
# Choose the threshold that maximizes F1-score on validation data.

val_acc, val_roc, val_pr, val_preds_argmax, val_probs, val_labels = evaluate(
    data.val_mask
)

thresholds = np.arange(0.05, 0.96, 0.01)

threshold_results = []

for threshold in thresholds:
    val_preds_thresholded = (val_probs >= threshold).astype(int)

    precision = precision_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    recall = recall_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    f1 = f1_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results = pd.DataFrame(threshold_results)

best_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold = float(best_row["threshold"])

print("\nBest threshold selected on validation set:")
print(best_row)

# Save threshold search results
threshold_results_path = GRAPHSAGE_COMMUNITY_AWARE_THRESHOLD_TABLE_PATH

threshold_results.to_csv(
    threshold_results_path,
    index=False
)

print("Saved threshold results:", threshold_results_path)

# ---------------------------------------------------------
# 12. Final test evaluation with selected threshold
# ---------------------------------------------------------

test_acc_argmax, test_roc, test_pr, test_preds_argmax, test_probs, test_labels = evaluate(
    data.test_mask
)

test_preds_thresholded = (test_probs >= best_threshold).astype(int)

print("\nFinal test results with validation-selected threshold")
print("Selected threshold:", best_threshold)
print("Test ROC-AUC:", test_roc)
print("Test PR-AUC:", test_pr)

print("\nClassification report:")
print(
    classification_report(
        test_labels,
        test_preds_thresholded,
        target_names=["normal", "fraud"],
        zero_division=0
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(test_labels, test_preds_thresholded))

# ---------------------------------------------------------
# 13. Predict fraud probability for every node
# ---------------------------------------------------------

model.eval()

with torch.no_grad():
    out = model(data.x, data.edge_index)
    probs = F.softmax(out, dim=1)
    fraud_probs = probs[:, 1].detach().cpu().numpy()

pred_df = pd.DataFrame({
    "node_id": np.arange(data.num_nodes),
    "fraud_probability": fraud_probs
})

node_mapping["node_id"] = node_mapping["node_id"].astype(int)

pred_df = pred_df.merge(
    node_mapping,
    on="node_id",
    how="left"
)

# Mark whether the node was labelled or unlabeled
y_cpu = data.y.detach().cpu().numpy()

pred_df["known_label"] = y_cpu
pred_df["is_labelled"] = pred_df["known_label"] != -1

# Add thresholded prediction
pred_df["predicted_label_thresholded"] = (
    pred_df["fraud_probability"] >= best_threshold
).astype(int)

pred_df["predicted_label_name"] = pred_df[
    "predicted_label_thresholded"
].map({
    0: "normal",
    1: "fraud"
})

pred_df = pred_df.sort_values(
    "fraud_probability",
    ascending=False
)

PRED_OUTPUT_PATH = GRAPHSAGE_COMMUNITY_AWARE_PREDICTIONS_PATH

pred_df.to_csv(
    PRED_OUTPUT_PATH,
    index=False
)

print("\nSaved predictions:")
print(PRED_OUTPUT_PATH)

display(pred_df.head(20))

Data(x=[1828910, 20], edge_index=[2, 6092594], edge_attr=[6092594, 2], num_nodes=1828910)
label
0    45000
1     4466
Name: count, dtype: int64
Total labelled nodes: 49466
Unlabelled nodes: 1779444
Normal labelled nodes: 45000
Fraud labelled nodes: 4466
Train nodes: 34626
Validation nodes: 7420
Test nodes: 7420
Using device: cuda
Class weights: tensor([0.1806, 1.8194], device='cuda:0')
Epoch 010 | Loss: 0.6354 | Train Acc: 0.7323 | Train PR-AUC: 0.2746 | Val Acc: 0.7311 | Val ROC-AUC: 0.8179 | Val PR-AUC: 0.2730
Saved best model.
Epoch 020 | Loss: 0.5433 | Train Acc: 0.7488 | Train PR-AUC: 0.4483 | Val Acc: 0.7456 | Val ROC-AUC: 0.8666 | Val PR-AUC: 0.4461
Saved best model.
Epoch 030 | Loss: 0.4763 | Train Acc: 0.7660 | Train PR-AUC: 0.5786 | Val Acc: 0.7633 | Val ROC-AUC: 0.9071 | Val PR-AUC: 0.5612
Saved best model.
Epoch 040 | Loss: 0.4302 | Train Acc: 0.7948 | Train PR-AUC: 0.6248 | Val Acc: 0.7972 | Val ROC-AUC: 0.9261 | Val PR-AUC: 0.6049
Saved best model.
Epoch 050 | Loss: 0.380

,node_id,fraud_probability,address,known_label,is_labelled,predicted_label_thresholded,predicted_label_name
27,27,0.999988,0x28c6c06298d514db089934071355e5743bf21d60,1,True,1,fraud
80,80,0.999988,0xa1abfa21f80ecf401bd41365adbb6fef6fefdf09,1,True,1,fraud
512,512,0.999986,0xf30ba13e4b04ce5dc4d254ae5fa95477800f0eb0,-1,False,1,fraud
46,46,0.999984,0x1ab4973a48dc892cd9971ece8e01dcc7688f8f23,1,True,1,fraud
5605,5605,0.999977,0xb5d85cbf7cb3ee0d56b3bb207d5fc4b82f43f511,1,True,1,fraud
1090,1090,0.999967,0xa9ac43f5b5e38155a288d1a01d2cbc4478e14573,-1,False,1,fraud
1806,1806,0.999965,0xa26148ae51fa8e787df319c04137602cc018b521,1,True,1,fraud
403,403,0.999961,0x2cff890f0378a11913b6129b2e97417a2c302680,-1,False,1,fraud
980,980,0.999958,0x974caa59e49682cda0ad2bbe82983419a2ecc400,1,True,1,fraud
246,246,0.999937,0xa03400e098f4421b34a3a44a1b4e571419517687,1,True,1,fraud


### Result comment: GraphSAGE with community-aware negative labels

The community-aware label setting makes the classification task harder and more realistic because the negative class is structurally more challenging: we purposely chose normal nodes that are more active and therefore more similar to the fraudulent ones, so they are less trivially separable. The model reaches ROC-AUC ≈ 0.967 and PR-AUC ≈ 0.775, with fraud precision around 0.59 and fraud recall around 0.80. Compared with the threshold-tuned baseline, performance decreases  This result is useful because it stress-tests the classifier under a label setting that better reflects the ambiguity of the full Ethereum graph.


### 11.1 Inspect high-risk predictions

Loads the prediction table from the alternative-label experiment for ranking and inspection.


In [ ]:

# ---------------------------------------------------------
# Path
# ---------------------------------------------------------

PRED_PATH = GRAPHSAGE_COMMUNITY_AWARE_PREDICTIONS_PATH

# ---------------------------------------------------------
# Load predictions
# ---------------------------------------------------------

pred_df = pd.read_csv(PRED_PATH)

print(pred_df.columns)

# ---------------------------------------------------------
# Count fraud predictions
# ---------------------------------------------------------

# If your file contains thresholded predictions
# created with the previous code:

fraud_count = (
    pred_df["predicted_label_thresholded"] == 1
).sum()

total_nodes = len(pred_df)

fraud_percentage = 100 * fraud_count / total_nodes

print("Total nodes:", total_nodes)
print("Predicted fraud nodes:", fraud_count)
print(f"Percentage predicted as fraud: {fraud_percentage:.4f}%")

Index(['node_id', 'fraud_probability', 'address', 'known_label', 'is_labelled',
       'predicted_label_thresholded', 'predicted_label_name'],
      dtype='object')
Total nodes: 1828910
Predicted fraud nodes: 15682
Percentage predicted as fraud: 0.8575%


## 12. Temporal motif feature extraction

Loads raw transactions and computes temporal motif counts such as out-stars, in-stars, chains, reciprocals, and approximate 3-cycles using a one-hour time window.


In [ ]:
# =========================================================
# Temporal motif counting for ALL graph nodes
# =========================================================
#
# This code computes simple directed temporal motifs
# over a rolling 1-hour window for every node.
#
# Motifs counted:
#
# 1. temporal_out_star
# 2. temporal_in_star
# 3. temporal_chain
# 4. temporal_reciprocal
# 5. temporal_cycle_3
# 6. temporal_fan_in_out
#
# The goal is NOT perfect motif mining efficiency,
# but rather:
#
# - obtain motif features for all nodes
# - test runtime feasibility
# - create node-level temporal motif vectors
#
# =========================================================

# ---------------------------------------------------------
# 1. Parameters
# ---------------------------------------------------------

TX_PATH = RAW_TX_PATH

OUTPUT_PATH = TEMPORAL_MOTIF_COUNTS_PATH

TIME_WINDOW_SECONDS = 3600   # 1 hour

# ---------------------------------------------------------
# 2. Load transactions
# ---------------------------------------------------------

print("Loading transactions...")

tx = pd.read_csv(
    TX_PATH,
    usecols=[
        "from_address",
        "to_address",
        "block_timestamp"
    ]
)

tx = tx.dropna()

tx["from_address"] = (
    tx["from_address"]
    .astype(str)
    .str.lower()
    .str.strip()
)

tx["to_address"] = (
    tx["to_address"]
    .astype(str)
    .str.lower()
    .str.strip()
)

tx["block_timestamp"] = pd.to_datetime(
    tx["block_timestamp"],
    errors="coerce",
    utc=True
)

tx = tx.dropna(subset=["block_timestamp"])

tx["timestamp_unix"] = (
    tx["block_timestamp"].astype("int64") // 10**9
)

tx["timestamp_unix"] = tx["timestamp_unix"].astype(np.int64)

print("Transactions loaded:", len(tx))

# ---------------------------------------------------------
# 3. Sort by time
# ---------------------------------------------------------

tx = tx.sort_values("timestamp_unix").reset_index(drop=True)


Loading transactions...
Transactions loaded: 6092594


### 12.1 Initialize temporal motif counters

Creates the per-node data structure that stores temporal motif counts.


In [ ]:

# ---------------------------------------------------------
# 4. Create temporal adjacency structures
# ---------------------------------------------------------

print("Creating temporal structures...")

# outgoing[src] -> list of (dst, timestamp)
outgoing = defaultdict(list)

# incoming[dst] -> list of (src, timestamp)
incoming = defaultdict(list)

for row in tqdm(
    tx.itertuples(index=False),
    total=len(tx),
    desc="Building temporal structures"
):
    src = row.from_address
    dst = row.to_address
    ts = row.timestamp_unix

    outgoing[src].append((dst, ts))
    incoming[dst].append((src, ts))

# ---------------------------------------------------------
# 5. Initialize motif counters
# ---------------------------------------------------------

motif_counts = defaultdict(
    lambda: {
        "temporal_out_star": 0,
        "temporal_in_star": 0,
        "temporal_chain": 0,
        "temporal_reciprocal": 0,
        "temporal_cycle_3": 0
    }
)


Creating temporal structures...


Building temporal structures:   0%|          | 0/6092594 [00:00<?, ?it/s]

### 12.2 Temporal out-star motifs

Counts repeated outgoing transactions from the same source within the time window.


In [ ]:

# ---------------------------------------------------------
# Temporal OUT-STAR with sliding window
# ---------------------------------------------------------

print("\nCounting temporal_out_star with sliding window...")

start = time.time()

for node, events in tqdm(
    outgoing.items(),
    total=len(outgoing),
    desc="OUT-STAR"
):
    events = sorted(events, key=lambda x: x[1])
    left = 0

    for right in range(len(events)):
        while events[right][1] - events[left][1] > TIME_WINDOW_SECONDS:
            left += 1

        window_size = right - left

        motif_counts[node]["temporal_out_star"] += window_size

end = time.time()
print(f"OUT-STAR completed in {(end-start)/60:.2f} minutes")


Counting temporal_out_star with sliding window...


OUT-STAR:   0%|          | 0/1625532 [00:00<?, ?it/s]

OUT-STAR completed in 0.15 minutes


### 12.3 Temporal in-star motifs

Counts repeated incoming transactions to the same destination within the time window.


In [ ]:

# ---------------------------------------------------------
# 7. Temporal IN-STAR
# ---------------------------------------------------------
#
# B -> A
# C -> A
#
# ---------------------------------------------------------

print("\nCounting temporal_in_star with sliding window...")

start = time.time()

for node, events in tqdm(
    incoming.items(),
    total=len(incoming),
    desc="IN-STAR"
):
    events = sorted(events, key=lambda x: x[1])
    left = 0

    for right in range(len(events)):
        while events[right][1] - events[left][1] > TIME_WINDOW_SECONDS:
            left += 1

        window_size = right - left

        # number of previous incoming events within 1h
        motif_counts[node]["temporal_in_star"] += window_size

end = time.time()
print(f"IN-STAR completed in {(end-start)/60:.2f} minutes")



Counting temporal_in_star with sliding window...


IN-STAR:   0%|          | 0/817935 [00:00<?, ?it/s]

IN-STAR completed in 0.13 minutes


### 12.4 Temporal chain motifs

Counts ordered two-step transaction chains that occur within the time window.


In [ ]:
# ---------------------------------------------------------
# Temporal CHAIN with sliding window
# ---------------------------------------------------------
#
# Pattern:
#   A -> B -> C
#
# The counted node is the middle node B.
# We count cases where B receives a transaction and then sends
# a transaction within TIME_WINDOW_SECONDS.
#
# This is much faster than the naive double loop.

print("\nCounting temporal_chain with sliding window...")

start = time.time()

for node in tqdm(
    set(incoming.keys()).intersection(set(outgoing.keys())),
    desc="CHAIN"
):
    in_events = sorted(incoming[node], key=lambda x: x[1])
    out_events = sorted(outgoing[node], key=lambda x: x[1])

    left = 0
    right = 0
    n_out = len(out_events)

    for src, t_in in in_events:

        # Move left pointer to first outgoing event after or at t_in
        while left < n_out and out_events[left][1] < t_in:
            left += 1

        # Move right pointer to first outgoing event outside the 1h window
        if right < left:
            right = left

        while right < n_out and out_events[right][1] - t_in <= TIME_WINDOW_SECONDS:
            right += 1

        # Outgoing events in [t_in, t_in + TIME_WINDOW_SECONDS]
        window_size = right - left

        motif_counts[node]["temporal_chain"] += window_size

end = time.time()

print(f"CHAIN completed in {(end-start)/60:.2f} minutes")



Counting temporal_chain with sliding window...


CHAIN:   0%|          | 0/614557 [00:00<?, ?it/s]

CHAIN completed in 0.12 minutes


### 12.5 Temporal reciprocal motifs

Counts bidirectional transaction patterns occurring within the time window.


In [ ]:

# ---------------------------------------------------------
# 9. Temporal RECIPROCAL
# ---------------------------------------------------------
#
# A -> B
# B -> A
#
# ---------------------------------------------------------

print("\nCounting temporal_reciprocal...")
start = time.time()
for src in tqdm(
    outgoing.keys(),
    total=len(outgoing),
    desc="RECIPROCAL"
):

    out_events = outgoing[src]

    for dst, t1 in out_events:

        reverse_events = outgoing.get(dst, [])

        for rev_dst, t2 in reverse_events:

            if rev_dst != src:
                continue

            if t2 < t1:
                continue

            if t2 - t1 > TIME_WINDOW_SECONDS:
                continue

            motif_counts[src]["temporal_reciprocal"] += 1
end = time.time()

print(f"RECIPROCAL completed in {(end-start)/60:.2f} minutes")



Counting temporal_reciprocal...


RECIPROCAL:   0%|          | 0/1625532 [00:00<?, ?it/s]

RECIPROCAL completed in 16.27 minutes


### 12.6 Approximate temporal 3-cycles

Estimates directed temporal cycles of length three and stores diagnostic information for the approximation.


In [ ]:


# ---------------------------------------------------------
# Sort temporal adjacency lists once
# ---------------------------------------------------------

print("Sorting temporal adjacency lists...")

for node in tqdm(outgoing.keys(), total=len(outgoing), desc="Sort outgoing"):
    outgoing[node].sort(key=lambda x: x[1])

for node in tqdm(incoming.keys(), total=len(incoming), desc="Sort incoming"):
    incoming[node].sort(key=lambda x: x[1])

out_sizes = pd.Series({node: len(events) for node, events in outgoing.items()})
in_sizes = pd.Series({node: len(events) for node, events in incoming.items()})

print("Outgoing size distribution:")
print(out_sizes.describe())

print("\nTop 20 outgoing nodes:")
print(out_sizes.sort_values(ascending=False).head(20))

print("\nIncoming size distribution:")
print(in_sizes.describe())

print("\nTop 20 incoming nodes:")
print(in_sizes.sort_values(ascending=False).head(20))

Sorting temporal adjacency lists...


Sort outgoing:   0%|          | 0/1625532 [00:00<?, ?it/s]

Sort incoming:   0%|          | 0/817935 [00:00<?, ?it/s]

Outgoing size distribution:
count    1.625532e+06
mean     3.748062e+00
std      1.147887e+02
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      4.614300e+04
dtype: float64

Top 20 outgoing nodes:
0x28c6c06298d514db089934071355e5743bf21d60    46143
0x559432e18b281731c054cd703d4b49872be4ed53    41888
0x4838b106fce9647bdf1e7877bf73ce8b0bad5f97    41087
0xf70da97812cb96acdf810712aa562db8dfa3dbef    38233
0x974caa59e49682cda0ad2bbe82983419a2ecc400    32590
0x46340b20830761efd32832a74d7169b29feb9758    27745
0x21a31ee1afc51d94c2efccaa2092ad1028285549    26828
0xdfd5293d8e347dfe59e90efd55b2956a1343963d    26495
0x264bd8291fae1d75db2c5f573b07faa6715997b5    24601
0x5050f69a9786f081509234f1a7f4684b5e5b76c9    24108
0xcf076007e6c36cfb1eb2cd36ad66405fec7d0b31    22129
0x1ab4973a48dc892cd9971ece8e01dcc7688f8f23    20927
0x0d0707963952f2fba59dd06f2b425ace40b492fe    18337
0xdadb0d80178819f2319190d340ce9a924f783711    18204
0x646c4fbdf82b5766c5eaf1fab9a

### 12.7 Save temporal cycle diagnostics

Exports diagnostic information from the temporal 3-cycle counting procedure.


In [ ]:
# ---------------------------------------------------------
# Temporal 3-CYCLE: exact for small nodes, estimated for large nodes
# ---------------------------------------------------------
#
# Pattern:
#   A -> B -> C -> A
#
# Conditions:
#   t1 <= t2 <= t3
#   t3 - t1 <= TIME_WINDOW_SECONDS
#
# Count is attributed to A.
#
# For small nodes:
#   exact count
#
# For large/high-cost nodes:
#   sample outgoing A -> B events and scale up the estimate
#
# The final value is stored in the SAME column:
#   motif_counts[A]["temporal_cycle_3"]
#
# No second feature column is created.

print("\nCounting temporal_cycle_3 with hybrid exact/estimated method...")

# ---------------------------------------------------------
# 1. Prepare sorted outgoing lists
# ---------------------------------------------------------

print("Preparing sorted outgoing lists...")

outgoing_sorted = {}
outgoing_times = {}

for node, events in tqdm(
    outgoing.items(),
    total=len(outgoing),
    desc="Preparing outgoing"
):
    events_sorted = sorted(events, key=lambda x: x[1])
    outgoing_sorted[node] = events_sorted
    outgoing_times[node] = [t for _, t in events_sorted]

# ---------------------------------------------------------
# 2. Parameters
# ---------------------------------------------------------

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# If A has more than this many outgoing events, approximate it
MAX_A_OUT_EVENTS_EXACT = 3000

# Number of A->B events sampled for approximate nodes
A_EDGE_SAMPLE_SIZE = 500

# If B or C has a very large outgoing list, sample candidates inside the valid time window
MAX_WINDOW_EVENTS_EXACT = 3000
WINDOW_SAMPLE_SIZE = 500

PRINT_EVERY_NODES = 5000
SLOW_NODE_SECONDS = 60

nodes_to_process = sorted(
    outgoing_sorted.keys(),
    key=lambda n: len(outgoing_sorted[n])
)

cycle_diagnostics = []

cycle_start_time = time.time()

# ---------------------------------------------------------
# 3. Helper: count cycles starting from one event A -> B at t1
# ---------------------------------------------------------

def count_cycles_from_edge(a, b, t1):
    """
    Counts or estimates the number of temporal cycles:

        A -> B -> C -> A

    starting from one fixed edge A -> B at time t1.

    Returns:
        estimated number of cycles starting from this edge
    """

    b_out_events = outgoing_sorted.get(b)

    if not b_out_events:
        return 0.0

    b_times = outgoing_times[b]

    # B -> C must occur in [t1, t1 + window]
    b_left = bisect.bisect_left(b_times, t1)
    b_right = bisect.bisect_right(
        b_times,
        t1 + TIME_WINDOW_SECONDS
    )

    b_window = b_out_events[b_left:b_right]

    if len(b_window) == 0:
        return 0.0

    # If the B-window is huge, sample it and scale up
    if len(b_window) > MAX_WINDOW_EVENTS_EXACT:
        sampled_b_window = random.sample(
            b_window,
            WINDOW_SAMPLE_SIZE
        )
        b_scale = len(b_window) / WINDOW_SAMPLE_SIZE
    else:
        sampled_b_window = b_window
        b_scale = 1.0

    cycle_count = 0.0

    for c, t2 in sampled_b_window:

        c_out_events = outgoing_sorted.get(c)

        if not c_out_events:
            continue

        c_times = outgoing_times[c]

        # C -> A must occur in [t2, t1 + window]
        c_left = bisect.bisect_left(c_times, t2)
        c_right = bisect.bisect_right(
            c_times,
            t1 + TIME_WINDOW_SECONDS
        )

        c_window = c_out_events[c_left:c_right]

        if len(c_window) == 0:
            continue

        # If C-window is huge, sample and scale up
        if len(c_window) > MAX_WINDOW_EVENTS_EXACT:
            sampled_c_window = random.sample(
                c_window,
                WINDOW_SAMPLE_SIZE
            )
            c_scale = len(c_window) / WINDOW_SAMPLE_SIZE
        else:
            sampled_c_window = c_window
            c_scale = 1.0

        hits = 0

        for dst, t3 in sampled_c_window:
            if dst == a:
                hits += 1

        cycle_count += b_scale * c_scale * hits

    return cycle_count

# ---------------------------------------------------------
# 4. Main loop
# ---------------------------------------------------------

for idx, a in enumerate(tqdm(
    nodes_to_process,
    total=len(nodes_to_process),
    desc="CYCLE_3"
)):
    node_start_time = time.time()

    a_out_events = outgoing_sorted[a]
    a_out_count = len(a_out_events)

    if idx % PRINT_EVERY_NODES == 0:
        print(
            f"\nProcessing idx={idx}/{len(nodes_to_process)} | "
            f"address={a} | "
            f"a_out={a_out_count} | "
            f"a_in={len(incoming.get(a, []))}"
        )

    # -----------------------------------------------------
    # Exact mode
    # -----------------------------------------------------

    if a_out_count <= MAX_A_OUT_EVENTS_EXACT:
        sampled_a_events = a_out_events
        a_scale = 1.0
        mode = "exact"

    # -----------------------------------------------------
    # Approximate mode
    # -----------------------------------------------------

    else:
        sample_size = min(A_EDGE_SAMPLE_SIZE, a_out_count)

        sampled_a_events = random.sample(
            a_out_events,
            sample_size
        )

        a_scale = a_out_count / sample_size
        mode = "estimated"

    cycle_count_for_a = 0.0

    for b, t1 in sampled_a_events:
        cycle_count_for_a += count_cycles_from_edge(a, b, t1)

    # Scale up if A was sampled
    cycle_count_for_a = cycle_count_for_a * a_scale

    # Store in the same motif column
    motif_counts[a]["temporal_cycle_3"] += cycle_count_for_a

    node_elapsed = time.time() - node_start_time

    if mode == "estimated" or node_elapsed > SLOW_NODE_SECONDS:
        cycle_diagnostics.append({
            "idx": idx,
            "address": a,
            "mode": mode,
            "seconds": node_elapsed,
            "a_out": a_out_count,
            "a_in": len(incoming.get(a, [])),
            "cycle_3_value": cycle_count_for_a
        })

        print(
            f"\n{mode.upper()} node | "
            f"idx={idx} | "
            f"address={a} | "
            f"time={node_elapsed:.2f}s | "
            f"a_out={a_out_count} | "
            f"cycle_3={cycle_count_for_a:.2f}"
        )

# ---------------------------------------------------------
# 5. Summary
# ---------------------------------------------------------

total_elapsed = time.time() - cycle_start_time

print(f"\nCYCLE_3 completed in {total_elapsed / 60:.2f} minutes")

cycle_diagnostics_df = pd.DataFrame(cycle_diagnostics)

if len(cycle_diagnostics_df) > 0:
    print("\nNodes estimated or slow:")
    display(
        cycle_diagnostics_df.sort_values(
            "cycle_3_value",
            ascending=False
        ).head(20)
    )

    diagnostics_path = CYCLE3_DIAGNOSTICS_PATH

    cycle_diagnostics_df.to_csv(
        diagnostics_path,
        index=False
    )

    print("Saved diagnostics:", diagnostics_path)


Counting temporal_cycle_3 with hybrid exact/estimated method...
Preparing sorted outgoing lists...


Preparing outgoing:   0%|          | 0/1625532 [00:00<?, ?it/s]

CYCLE_3:   0%|          | 0/1625532 [00:00<?, ?it/s]


Processing idx=0/1625532 | address=0x89211817d39ca280220727bf0a1b1cbb2178915d | a_out=1 | a_in=0

Processing idx=5000/1625532 | address=0xd3227d1abcb5fbb17cc491a8ec5f12b03fb2df86 | a_out=1 | a_in=1

Processing idx=10000/1625532 | address=0x72b1121248a25ff3bd9e4a2fcf6b5e8bfc39c44e | a_out=1 | a_in=0

Processing idx=15000/1625532 | address=0x1bce7732df3769884a04fd6c45136a499b4c0148 | a_out=1 | a_in=0

Processing idx=20000/1625532 | address=0x5dbc65a9ab5b700a7247993cfeed07a7b8f22f25 | a_out=1 | a_in=1

Processing idx=25000/1625532 | address=0x58de44b5cc742f44bab0ca0a0162b0a54c629db8 | a_out=1 | a_in=1

Processing idx=30000/1625532 | address=0xf5d74c50bdceeed7a116550146ebefccb946b1f9 | a_out=1 | a_in=0

Processing idx=35000/1625532 | address=0xd610c2c3f0160e46ab9ee5028cbfbf6908d0f985 | a_out=1 | a_in=0

Processing idx=40000/1625532 | address=0x47e06fec909ce3ed83c566eb3cfdbdd519069b26 | a_out=1 | a_in=0

Processing idx=45000/1625532 | address=0xa1cd4d12147091678e4fc749f903838810814d04 | a_

,idx,address,mode,seconds,a_out,a_in,cycle_3_value
129,1625511,0x21b92a8fe81f6f300c03d8a3a403aac32c2facc2,estimated,43.642977,15273,15273,6.228800e+10
139,1625521,0xcf076007e6c36cfb1eb2cd36ad66405fec7d0b31,estimated,9.079580,22129,4999,2.023956e+10
0,1625334,0x40a9f78879595e961fda688c69537c3529777426,exact,96.129728,2107,2131,1.774752e+09
49,1625431,0x0067cc2416f792cf0ec5629f5506324ac508f859,estimated,0.002858,4331,2489,1.020297e+05
33,1625415,0x391e7c679d29bd940d63be94ad22a25d25b5a604,estimated,0.091524,3669,3203,8.321292e+03
133,1625515,0x5babe600b9fcd5fb7b66c0611bf4896d967b23a1,estimated,0.002985,16193,2173,7.384008e+03
61,1625443,0x555ce236c0220695b68341bc48c68d52210cc35b,estimated,0.001605,4663,323,6.490896e+03
79,1625461,0x307576dd4f73f91bb8c4a2edb762938e8e067d31,estimated,0.001868,5657,20621,3.439456e+03
118,1625500,0xb23360ccdd9ed1b15d45e5d3824bb409c8d7c460,estimated,0.002810,10867,54,7.606900e+02
54,1625436,0xdfaa75323fb721e5f29d43859390f62cc4b600b8,estimated,0.002075,4447,7325,5.336400e+02


Saved diagnostics: /content/drive/MyDrive/Graph_Mining_Project/outputs/08_diagnostics/temporal_cycle3_estimation_diagnostics.csv


### 12.8 Save temporal motif feature table

Exports the node-level temporal motif counts for all observed addresses.


In [ ]:


# ---------------------------------------------------------
# 12. Convert to dataframe
# ---------------------------------------------------------

print("\nCreating dataframe...")

rows = []

for node, counts in motif_counts.items():

    row = {
        "address": node
    }

    row.update(counts)

    rows.append(row)

motif_df = pd.DataFrame(rows)

# ---------------------------------------------------------
# 13. Log-transform motif counts
# ---------------------------------------------------------
# Remove unused motif column
motif_df = motif_df.drop(
    columns=["temporal_fan_in_out"],
    errors="ignore"
)

motif_cols = [
    "temporal_out_star",
    "temporal_in_star",
    "temporal_chain",
    "temporal_reciprocal",
    "temporal_cycle_3"
]

for col in motif_cols:
    motif_df[col] = np.log1p(motif_df[col])

# ---------------------------------------------------------
# 14. Save
# ---------------------------------------------------------

motif_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved motif features:")
print(OUTPUT_PATH)

print("\nShape:")
print(motif_df.shape)

display(motif_df.head())


Creating dataframe...

Saved motif features:
/content/drive/MyDrive/Graph_Mining_Project/outputs/04_temporal_motifs/temporal_motif_counts_1h_all_nodes.csv

Shape:
(1828910, 6)


,address,temporal_out_star,temporal_in_star,temporal_chain,temporal_reciprocal,temporal_cycle_3
0,0x06354ce08f96bc961b9f5110d513e1bedf256ce6,6.342121,6.408529,6.682109,0.000000,0.000000
1,0xa1abfa21f80ecf401bd41365adbb6fef6fefdf09,14.613776,13.950441,14.247688,3.555348,4.427215
2,0x56eddb7aa87536c09ccc2793473599fd21a8b17f,14.772671,1.386294,7.408531,0.000000,0.000000
3,0xab97925eb84fe0260779f58b7cb08d77dcb1ee2b,13.885771,0.000000,6.066108,0.000000,0.000000
4,0x35e2ef3afb4a87607351bac1ca16c1510ba94398,2.772589,0.000000,1.386294,0.000000,0.000000


## 13. Graph reconstruction with temporal motif features

Defines paths for augmenting the original node feature table with temporal motif counts and saving a new PyTorch Geometric graph.


In [ ]:

GRAPH_PATH = BASIC_GRAPH_PATH
NODE_FEATURES_PATH = NODE_BASIC_FEATURES_PATH
NODE_MAPPING_PATH = NODE_MAPPING_FILE_PATH
MOTIF_PATH = TEMPORAL_MOTIF_COUNTS_PATH

NEW_FEATURES_PATH = NODE_FEATURES_WITH_TEMPORAL_MOTIFS_PATH
NEW_GRAPH_PATH = TEMPORAL_MOTIF_GRAPH_PATH

### 13.1 Load base graph, features, mapping, and motifs

Loads the previously saved graph artifacts and the temporal motif table.


In [ ]:
# ---------------------------------------------------------
# 1. Load files
# ---------------------------------------------------------

data = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

node_features = pd.read_csv(NODE_FEATURES_PATH)
node_mapping = pd.read_csv(NODE_MAPPING_PATH)
motifs = pd.read_csv(MOTIF_PATH)

# ---------------------------------------------------------
# 2. Standardize identifiers
# ---------------------------------------------------------

if "node_id" not in node_features.columns:
    node_features["node_id"] = node_features.index

node_features["node_id"] = node_features["node_id"].astype(int)

node_mapping["node_id"] = node_mapping["node_id"].astype(int)
node_mapping["address"] = (
    node_mapping["address"]
    .astype(str)
    .str.lower()
    .str.strip()
)

motifs["address"] = (
    motifs["address"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Attach node_id to motif table
motifs = motifs.merge(
    node_mapping[["address", "node_id"]],
    on="address",
    how="inner"
)

print("Motif rows matched to graph:", len(motifs))

Motif rows matched to graph: 1828910


### 13.2 Align motif features to node IDs

Maps motif rows to graph node IDs so the temporal features align with the graph feature matrix.


In [ ]:
# ---------------------------------------------------------
# 3. Merge motif features into node features
# ---------------------------------------------------------

motif_cols = [
    "temporal_out_star",
    "temporal_in_star",
    "temporal_chain",
    "temporal_reciprocal",
    "temporal_cycle_3"
]

# Safety: keep only motif columns that exist
motif_cols = [col for col in motif_cols if col in motifs.columns]

node_features_with_motifs = node_features.merge(
    motifs[["node_id"] + motif_cols],
    on="node_id",
    how="left"
)

# Missing motif values mean no counted motifs for that node
for col in motif_cols:
    node_features_with_motifs[col] = (
        node_features_with_motifs[col]
        .fillna(0.0)
        .replace([np.inf, -np.inf], 0.0)
    )

print("Original feature shape:", node_features.shape)
print("New feature shape:", node_features_with_motifs.shape)
print("Added motif features:", motif_cols)

Original feature shape: (1828910, 21)
New feature shape: (1828910, 26)
Added motif features: ['temporal_out_star', 'temporal_in_star', 'temporal_chain', 'temporal_reciprocal', 'temporal_cycle_3']


### 13.3 Merge temporal motifs into node features

Combines basic engineered node features with temporal motif counts, filling missing motif values with zeros.


In [ ]:
# ---------------------------------------------------------
# 4. Rebuild data.x
# ---------------------------------------------------------

exclude_cols = ["node_id", "address"]

feature_cols = [
    col for col in node_features_with_motifs.columns
    if col not in exclude_cols
]

x = torch.tensor(
    node_features_with_motifs[feature_cols].values,
    dtype=torch.float
)

data.x = x

print(data)
print("New data.x shape:", data.x.shape)
print("Number of feature columns:", len(feature_cols))

Data(x=[1828910, 25], edge_index=[2, 6092594], edge_attr=[6092594, 2], num_nodes=1828910)
New data.x shape: torch.Size([1828910, 25])
Number of feature columns: 25


### 13.4 Save motif-augmented graph artifacts

Exports the motif-augmented feature table and the corresponding PyTorch Geometric graph.


In [ ]:
# ---------------------------------------------------------
# 5. Save updated graph and feature table
# ---------------------------------------------------------

node_features_with_motifs.to_csv(
    NEW_FEATURES_PATH,
    index=False
)

torch.save(
    data,
    NEW_GRAPH_PATH
)

print("Saved:")
print(NEW_FEATURES_PATH)
print(NEW_GRAPH_PATH)

Saved:
/content/drive/MyDrive/Graph_Mining_Project/outputs/04_temporal_motifs/ethereum_node_features_with_temporal_motifs.csv
/content/drive/MyDrive/Graph_Mining_Project/outputs/04_temporal_motifs/ethereum_pyg_graph_with_temporal_motifs.pt


## 14. GraphSAGE with motif-augmented features

Trains GraphSAGE on the graph whose node features include temporal motif counts and applies validation-threshold selection.


In [ ]:
# =========================================================
# GraphSAGE semi-supervised node classification
# with validation-selected fraud threshold AND MOTIF
# =========================================================

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------

GRAPH_PATH = TEMPORAL_MOTIF_GRAPH_PATH
LABEL_PATH = COMMUNITY_AWARE_LABELS_PATH
NODE_MAPPING_PATH = NODE_MAPPING_FILE_PATH

# ---------------------------------------------------------
# 2. Load graph and labels
# ---------------------------------------------------------

data = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

labels_df = pd.read_csv(LABEL_PATH)
node_mapping = pd.read_csv(NODE_MAPPING_PATH)

labels_df["node_id"] = labels_df["node_id"].astype(int)
labels_df["label"] = labels_df["label"].astype(int)

print(data)
print(labels_df["label"].value_counts())
print("Total labelled nodes:", len(labels_df))

# ---------------------------------------------------------
# 3. Create y vector
# ---------------------------------------------------------
# -1 = unlabeled
#  0 = normal
#  1 = fraud

num_nodes = data.num_nodes

y = torch.full(
    (num_nodes,),
    -1,
    dtype=torch.long
)

labelled_node_ids = labels_df["node_id"].values
label_values = labels_df["label"].values

y[labelled_node_ids] = torch.tensor(
    label_values,
    dtype=torch.long
)

data.y = y

print("Unlabelled nodes:", (data.y == -1).sum().item())
print("Normal labelled nodes:", (data.y == 0).sum().item())
print("Fraud labelled nodes:", (data.y == 1).sum().item())

# ---------------------------------------------------------
# 4. Train / validation / test split
# ---------------------------------------------------------

train_ids, temp_ids = train_test_split(
    labelled_node_ids,
    test_size=0.30,
    random_state=42,
    stratify=label_values
)

temp_labels = y[temp_ids].numpy()

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_ids] = True
val_mask[val_ids] = True
test_mask[test_ids] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print("Train nodes:", data.train_mask.sum().item())
print("Validation nodes:", data.val_mask.sum().item())
print("Test nodes:", data.test_mask.sum().item())

# ---------------------------------------------------------
# Select reusable GraphSAGE architecture
# ---------------------------------------------------------

GraphSAGE = GraphSAGEOneHidden

# ---------------------------------------------------------
# 6. Device, model, optimizer
# ---------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

data = data.to(device)

model = GraphSAGE(
    in_channels=data.x.shape[1],
    hidden_channels=64,
    out_channels=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)


# ---------------------------------------------------------
# 7. Class weights for imbalance
# ---------------------------------------------------------

train_labels = data.y[data.train_mask]

num_normal = (train_labels == 0).sum().item()
num_fraud = (train_labels == 1).sum().item()

class_weights = torch.tensor(
    [
        1.0 / num_normal,
        1.0 / num_fraud
    ],
    dtype=torch.float,
    device=device
)

class_weights = class_weights / class_weights.sum() * 2

print("Class weights:", class_weights)
# ---------------------------------------------------------
# Reuse standard training and evaluation functions
# ---------------------------------------------------------

# train_one_epoch() and evaluate() are defined in the shared utilities section.

# ---------------------------------------------------------
# 9. Train model
# ---------------------------------------------------------

EPOCHS = 150

best_val_pr_auc = 0
best_model_path = BEST_GRAPHSAGE_TEMPORAL_MOTIF_1H_PATH

for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch()

    if epoch % 10 == 0:
        train_acc, train_roc, train_pr, _, _, _ = evaluate(data.train_mask)
        val_acc, val_roc, val_pr, _, _, _ = evaluate(data.val_mask)

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Train PR-AUC: {train_pr:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val ROC-AUC: {val_roc:.4f} | "
            f"Val PR-AUC: {val_pr:.4f}"
        )

        if val_pr > best_val_pr_auc:
            best_val_pr_auc = val_pr

            torch.save(
                model.state_dict(),
                best_model_path
            )

            print("Saved best model.")

# ---------------------------------------------------------
# 10. Load best model
# ---------------------------------------------------------

model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

# ---------------------------------------------------------
# 11. Select best fraud threshold on validation set
# ---------------------------------------------------------
# Choose the threshold that maximizes F1-score on validation data.

val_acc, val_roc, val_pr, val_preds_argmax, val_probs, val_labels = evaluate(
    data.val_mask
)

thresholds = np.arange(0.05, 0.96, 0.01)

threshold_results = []

for threshold in thresholds:
    val_preds_thresholded = (val_probs >= threshold).astype(int)

    precision = precision_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    recall = recall_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    f1 = f1_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results = pd.DataFrame(threshold_results)

best_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold = float(best_row["threshold"])

print("\nBest threshold selected on validation set:")
print(best_row)

# Save threshold search results
threshold_results_path = GRAPHSAGE_TEMPORAL_MOTIF_1H_THRESHOLD_TABLE_PATH

threshold_results.to_csv(
    threshold_results_path,
    index=False
)

print("Saved threshold results:", threshold_results_path)

# ---------------------------------------------------------
# 12. Final test evaluation with selected threshold
# ---------------------------------------------------------

test_acc_argmax, test_roc, test_pr, test_preds_argmax, test_probs, test_labels = evaluate(
    data.test_mask
)

test_preds_thresholded = (test_probs >= best_threshold).astype(int)

print("\nFinal test results with validation-selected threshold")
print("Selected threshold:", best_threshold)
print("Test ROC-AUC:", test_roc)
print("Test PR-AUC:", test_pr)

print("\nClassification report:")
print(
    classification_report(
        test_labels,
        test_preds_thresholded,
        target_names=["normal", "fraud"],
        zero_division=0
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(test_labels, test_preds_thresholded))

# ---------------------------------------------------------
# 13. Predict fraud probability for every node
# ---------------------------------------------------------

model.eval()

with torch.no_grad():
    out = model(data.x, data.edge_index)
    probs = F.softmax(out, dim=1)
    fraud_probs = probs[:, 1].detach().cpu().numpy()

pred_df = pd.DataFrame({
    "node_id": np.arange(data.num_nodes),
    "fraud_probability": fraud_probs
})

node_mapping["node_id"] = node_mapping["node_id"].astype(int)

pred_df = pred_df.merge(
    node_mapping,
    on="node_id",
    how="left"
)

# Mark whether the node was labelled or unlabeled
y_cpu = data.y.detach().cpu().numpy()

pred_df["known_label"] = y_cpu
pred_df["is_labelled"] = pred_df["known_label"] != -1

# Add thresholded prediction
pred_df["predicted_label_thresholded"] = (
    pred_df["fraud_probability"] >= best_threshold
).astype(int)

pred_df["predicted_label_name"] = pred_df[
    "predicted_label_thresholded"
].map({
    0: "normal",
    1: "fraud"
})

pred_df = pred_df.sort_values(
    "fraud_probability",
    ascending=False
)

PRED_OUTPUT_PATH = GRAPHSAGE_TEMPORAL_MOTIF_1H_PREDICTIONS_PATH

pred_df.to_csv(
    PRED_OUTPUT_PATH,
    index=False
)

print("\nSaved predictions:")
print(PRED_OUTPUT_PATH)

display(pred_df.head(20))

Data(x=[1828910, 25], edge_index=[2, 6092594], edge_attr=[6092594, 2], num_nodes=1828910)
label
0    45000
1     4466
Name: count, dtype: int64
Total labelled nodes: 49466
Unlabelled nodes: 1779444
Normal labelled nodes: 45000
Fraud labelled nodes: 4466
Train nodes: 34626
Validation nodes: 7420
Test nodes: 7420
Using device: cuda
Class weights: tensor([0.1806, 1.8194], device='cuda:0')
Epoch 010 | Loss: 0.5517 | Train Acc: 0.7617 | Train PR-AUC: 0.5861 | Val Acc: 0.7606 | Val ROC-AUC: 0.8807 | Val PR-AUC: 0.5822
Saved best model.
Epoch 020 | Loss: 0.4521 | Train Acc: 0.8042 | Train PR-AUC: 0.6677 | Val Acc: 0.8026 | Val ROC-AUC: 0.9193 | Val PR-AUC: 0.6485
Saved best model.
Epoch 030 | Loss: 0.3797 | Train Acc: 0.8520 | Train PR-AUC: 0.6840 | Val Acc: 0.8524 | Val ROC-AUC: 0.9379 | Val PR-AUC: 0.6645
Saved best model.
Epoch 040 | Loss: 0.3460 | Train Acc: 0.8713 | Train PR-AUC: 0.6849 | Val Acc: 0.8710 | Val ROC-AUC: 0.9427 | Val PR-AUC: 0.6697
Saved best model.
Epoch 050 | Loss: 0.313

,node_id,fraud_probability,address,known_label,is_labelled,predicted_label_thresholded,predicted_label_name
1806,1806,0.999998,0xa26148ae51fa8e787df319c04137602cc018b521,1,True,1,fraud
80,80,0.999997,0xa1abfa21f80ecf401bd41365adbb6fef6fefdf09,1,True,1,fraud
980,980,0.999997,0x974caa59e49682cda0ad2bbe82983419a2ecc400,1,True,1,fraud
27,27,0.999997,0x28c6c06298d514db089934071355e5743bf21d60,1,True,1,fraud
46,46,0.999997,0x1ab4973a48dc892cd9971ece8e01dcc7688f8f23,1,True,1,fraud
18224,18224,0.999993,0xeb943c230218e0da7b2ca18b81f7eb7fbbbe9665,1,True,1,fraud
1090,1090,0.999993,0xa9ac43f5b5e38155a288d1a01d2cbc4478e14573,-1,False,1,fraud
933,933,0.999992,0x307576dd4f73f91bb8c4a2edb762938e8e067d31,-1,False,1,fraud
403,403,0.999991,0x2cff890f0378a11913b6129b2e97417a2c302680,-1,False,1,fraud
512,512,0.999991,0xf30ba13e4b04ce5dc4d254ae5fa95477800f0eb0,-1,False,1,fraud


### Result comment: GraphSAGE with temporal motif-augmented node features

Adding temporal motif counts directly to the GraphSAGE input gives ROC-AUC ≈ 0.974 and PR-AUC ≈ 0.816. Fraud precision is about 0.70 and fraud recall about 0.80. Relative to the community-aware GraphSAGE model, this improves precision and keeps recall at a similar level, showing that temporal motifs add useful behavioural signal. However, direct fusion is still limited because motif counts are passed through GraphSAGE message passing together with the static/basic features, which may blur their temporal meaning.


## 15. Two-hidden-layer GraphSAGE

Extends the GraphSAGE architecture to two hidden layers for the motif-augmented feature setting.


In [ ]:
# =========================================================
# GraphSAGE semi-supervised node classification
# with validation-selected fraud threshold AND MOTIF
# =========================================================

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------

GRAPH_PATH = TEMPORAL_MOTIF_GRAPH_PATH
LABEL_PATH = COMMUNITY_AWARE_LABELS_PATH
NODE_MAPPING_PATH = NODE_MAPPING_FILE_PATH

# ---------------------------------------------------------
# 2. Load graph and labels
# ---------------------------------------------------------

data = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

labels_df = pd.read_csv(LABEL_PATH)
node_mapping = pd.read_csv(NODE_MAPPING_PATH)

labels_df["node_id"] = labels_df["node_id"].astype(int)
labels_df["label"] = labels_df["label"].astype(int)

print(data)
print(labels_df["label"].value_counts())
print("Total labelled nodes:", len(labels_df))

# ---------------------------------------------------------
# 3. Create y vector
# ---------------------------------------------------------
# -1 = unlabeled
#  0 = normal
#  1 = fraud

num_nodes = data.num_nodes

y = torch.full(
    (num_nodes,),
    -1,
    dtype=torch.long
)

labelled_node_ids = labels_df["node_id"].values
label_values = labels_df["label"].values

y[labelled_node_ids] = torch.tensor(
    label_values,
    dtype=torch.long
)

data.y = y

print("Unlabelled nodes:", (data.y == -1).sum().item())
print("Normal labelled nodes:", (data.y == 0).sum().item())
print("Fraud labelled nodes:", (data.y == 1).sum().item())

# ---------------------------------------------------------
# 4. Train / validation / test split
# ---------------------------------------------------------

train_ids, temp_ids = train_test_split(
    labelled_node_ids,
    test_size=0.30,
    random_state=42,
    stratify=label_values
)

temp_labels = y[temp_ids].numpy()

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_ids] = True
val_mask[val_ids] = True
test_mask[test_ids] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print("Train nodes:", data.train_mask.sum().item())
print("Validation nodes:", data.val_mask.sum().item())
print("Test nodes:", data.test_mask.sum().item())

# ---------------------------------------------------------
# Select reusable GraphSAGE architecture
# ---------------------------------------------------------

GraphSAGE = GraphSAGETwoHidden

# ---------------------------------------------------------
# 6. Device, model, optimizer
# ---------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

data = data.to(device)

model = GraphSAGE(
    in_channels=data.x.shape[1],
    hidden_channels=64,
    out_channels=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)


# ---------------------------------------------------------
# 7. Class weights for imbalance
# ---------------------------------------------------------

train_labels = data.y[data.train_mask]

num_normal = (train_labels == 0).sum().item()
num_fraud = (train_labels == 1).sum().item()

class_weights = torch.tensor(
    [
        1.0 / num_normal,
        1.0 / num_fraud
    ],
    dtype=torch.float,
    device=device
)

class_weights = class_weights / class_weights.sum() * 2

print("Class weights:", class_weights)
# ---------------------------------------------------------
# Reuse standard training and evaluation functions
# ---------------------------------------------------------

# train_one_epoch() and evaluate() are defined in the shared utilities section.

# ---------------------------------------------------------
# 9. Train model
# ---------------------------------------------------------

EPOCHS = 150

best_val_pr_auc = 0
best_model_path = BEST_GRAPHSAGE_TEMPORAL_MOTIF_2H_PATH
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch()

    if epoch % 10 == 0:
        train_acc, train_roc, train_pr, _, _, _ = evaluate(data.train_mask)
        val_acc, val_roc, val_pr, _, _, _ = evaluate(data.val_mask)

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Train PR-AUC: {train_pr:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val ROC-AUC: {val_roc:.4f} | "
            f"Val PR-AUC: {val_pr:.4f}"
        )

        if val_pr > best_val_pr_auc:
            best_val_pr_auc = val_pr

            torch.save(
                model.state_dict(),
                best_model_path
            )

            print("Saved best model.")

# ---------------------------------------------------------
# 10. Load best model
# ---------------------------------------------------------

model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

# ---------------------------------------------------------
# 11. Select best fraud threshold on validation set
# ---------------------------------------------------------
# Choose the threshold that maximizes F1-score on validation data.

val_acc, val_roc, val_pr, val_preds_argmax, val_probs, val_labels = evaluate(
    data.val_mask
)

thresholds = np.arange(0.05, 0.96, 0.01)

threshold_results = []

for threshold in thresholds:
    val_preds_thresholded = (val_probs >= threshold).astype(int)

    precision = precision_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    recall = recall_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    f1 = f1_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results = pd.DataFrame(threshold_results)

best_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold = float(best_row["threshold"])

print("\nBest threshold selected on validation set:")
print(best_row)

# Save threshold search results
threshold_results_path = GRAPHSAGE_TEMPORAL_MOTIF_2H_THRESHOLD_TABLE_PATH

threshold_results.to_csv(
    threshold_results_path,
    index=False
)

print("Saved threshold results:", threshold_results_path)

# ---------------------------------------------------------
# 12. Final test evaluation with selected threshold
# ---------------------------------------------------------

test_acc_argmax, test_roc, test_pr, test_preds_argmax, test_probs, test_labels = evaluate(
    data.test_mask
)

test_preds_thresholded = (test_probs >= best_threshold).astype(int)

print("\nFinal test results with validation-selected threshold")
print("Selected threshold:", best_threshold)
print("Test ROC-AUC:", test_roc)
print("Test PR-AUC:", test_pr)

print("\nClassification report:")
print(
    classification_report(
        test_labels,
        test_preds_thresholded,
        target_names=["normal", "fraud"],
        zero_division=0
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(test_labels, test_preds_thresholded))

# ---------------------------------------------------------
# 13. Predict fraud probability for every node
# ---------------------------------------------------------

model.eval()

with torch.no_grad():
    out = model(data.x, data.edge_index)
    probs = F.softmax(out, dim=1)
    fraud_probs = probs[:, 1].detach().cpu().numpy()

pred_df = pd.DataFrame({
    "node_id": np.arange(data.num_nodes),
    "fraud_probability": fraud_probs
})

node_mapping["node_id"] = node_mapping["node_id"].astype(int)

pred_df = pred_df.merge(
    node_mapping,
    on="node_id",
    how="left"
)

# Mark whether the node was labelled or unlabeled
y_cpu = data.y.detach().cpu().numpy()

pred_df["known_label"] = y_cpu
pred_df["is_labelled"] = pred_df["known_label"] != -1

# Add thresholded prediction
pred_df["predicted_label_thresholded"] = (
    pred_df["fraud_probability"] >= best_threshold
).astype(int)

pred_df["predicted_label_name"] = pred_df[
    "predicted_label_thresholded"
].map({
    0: "normal",
    1: "fraud"
})

pred_df = pred_df.sort_values(
    "fraud_probability",
    ascending=False
)

PRED_OUTPUT_PATH = GRAPHSAGE_TEMPORAL_MOTIF_2H_PREDICTIONS_PATH

pred_df.to_csv(
    PRED_OUTPUT_PATH,
    index=False
)

print("\nSaved predictions:")
print(PRED_OUTPUT_PATH)

display(pred_df.head(20))

Data(x=[1828910, 25], edge_index=[2, 6092594], edge_attr=[6092594, 2], num_nodes=1828910)
label
0    45000
1     4466
Name: count, dtype: int64
Total labelled nodes: 49466
Unlabelled nodes: 1779444
Normal labelled nodes: 45000
Fraud labelled nodes: 4466
Train nodes: 34626
Validation nodes: 7420
Test nodes: 7420
Using device: cuda
Class weights: tensor([0.1806, 1.8194], device='cuda:0')
Epoch 010 | Loss: 0.6451 | Train Acc: 0.4647 | Train PR-AUC: 0.3413 | Val Acc: 0.4617 | Val ROC-AUC: 0.8487 | Val PR-AUC: 0.3408
Saved best model.
Epoch 020 | Loss: 0.5343 | Train Acc: 0.7599 | Train PR-AUC: 0.5895 | Val Acc: 0.7592 | Val ROC-AUC: 0.8989 | Val PR-AUC: 0.5868
Saved best model.
Epoch 030 | Loss: 0.4433 | Train Acc: 0.8161 | Train PR-AUC: 0.6559 | Val Acc: 0.8131 | Val ROC-AUC: 0.9304 | Val PR-AUC: 0.6473
Saved best model.
Epoch 040 | Loss: 0.3637 | Train Acc: 0.8840 | Train PR-AUC: 0.6951 | Val Acc: 0.8815 | Val ROC-AUC: 0.9434 | Val PR-AUC: 0.6888
Saved best model.
Epoch 050 | Loss: 0.322

,node_id,fraud_probability,address,known_label,is_labelled,predicted_label_thresholded,predicted_label_name
137667,137667,0.999790,0xfdeced66af5596e1a4567b52a71060e3ba470eb3,1,True,1,fraud
1806,1806,0.999789,0xa26148ae51fa8e787df319c04137602cc018b521,1,True,1,fraud
980,980,0.999641,0x974caa59e49682cda0ad2bbe82983419a2ecc400,1,True,1,fraud
27,27,0.999543,0x28c6c06298d514db089934071355e5743bf21d60,1,True,1,fraud
239339,239339,0.999506,0xa26e73c8e9507d50bf808b7a2ca9d5de4fcc4a04,1,True,1,fraud
46,46,0.999403,0x1ab4973a48dc892cd9971ece8e01dcc7688f8f23,1,True,1,fraud
5605,5605,0.999394,0xb5d85cbf7cb3ee0d56b3bb207d5fc4b82f43f511,1,True,1,fraud
4904,4904,0.999391,0x62fdfeeca74c7f1c6dfe88777271f76bd75576a7,1,True,1,fraud
18224,18224,0.999335,0xeb943c230218e0da7b2ca18b81f7eb7fbbbe9665,1,True,1,fraud
80,80,0.999241,0xa1abfa21f80ecf401bd41365adbb6fef6fefdf09,1,True,1,fraud


### Result comment: two-hidden-layer GraphSAGE with temporal motifs

The two-hidden-layer temporal-motif GraphSAGE improves over the one-hidden-layer version. PR-AUC rises from about 0.816 to about 0.849, fraud precision increases from about 0.70 to about 0.75, and fraud recall increases from about 0.80 to about 0.84. This suggests that the deeper GraphSAGE encoder better combines graph-neighbourhood information with motif-augmented features. The improvement is meaningful, but this direct-fusion approach still remains weaker than the best threshold-tuned baseline and weaker than the temporal motif late-fusion MLP.


## 16. Snap ML motif features only: one hidden layer

Trains a GraphSAGE classifier using only the Snap ML motif-derived node features as the input feature matrix.


In [ ]:
# =========================================================
# GraphSAGE semi-supervised node classification
# using ONLY Snap ML motif/node features
# 1 hidden layer
# =========================================================

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------

GRAPH_PATH = BASIC_GRAPH_PATH
LABEL_PATH = COMMUNITY_AWARE_LABELS_PATH
NODE_MAPPING_PATH = NODE_MAPPING_FILE_PATH

SNAP_FEATURE_PATH = SNAP_ML_FEATURE_PATH

BEST_MODEL_PATH = BEST_GRAPHSAGE_SNAP_ONLY_1H_PATH

PRED_OUTPUT_PATH = GRAPHSAGE_SNAP_ONLY_1H_PREDICTIONS_PATH

# ---------------------------------------------------------
# 2. Load graph, labels, node mapping, and Snap ML features
# ---------------------------------------------------------

data = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

labels_df = pd.read_csv(LABEL_PATH)
node_mapping = pd.read_csv(NODE_MAPPING_PATH)
snap_features = pd.read_csv(SNAP_FEATURE_PATH)

labels_df["node_id"] = labels_df["node_id"].astype(int)
labels_df["label"] = labels_df["label"].astype(int)

node_mapping["node_id"] = node_mapping["node_id"].astype(int)

print("Loaded graph:")
print(data)

print("\nLabels:")
print(labels_df["label"].value_counts())

print("\nSnap ML feature file shape:")
print(snap_features.shape)
print(snap_features.head())

# ---------------------------------------------------------
# 3. Prepare Snap ML node features
# ---------------------------------------------------------
# The Snap ML file uses vertex_id.
# We assume vertex_id corresponds to the same node_id used in the graph.

if "vertex_id" not in snap_features.columns:
    raise ValueError("Expected column 'vertex_id' in Snap ML feature file.")

snap_features = snap_features.rename(columns={"vertex_id": "node_id"})
snap_features["node_id"] = snap_features["node_id"].astype(int)

# Remove duplicated node rows if any
snap_features = snap_features.drop_duplicates(subset="node_id")

# Identify feature columns
id_cols = ["node_id", "address"]
feature_cols = [
    c for c in snap_features.columns
    if c not in id_cols
]

print("\nNumber of Snap ML feature columns:", len(feature_cols))

# Clean numeric values
for col in feature_cols:
    snap_features[col] = pd.to_numeric(
        snap_features[col],
        errors="coerce"
    )

snap_features[feature_cols] = (
    snap_features[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

# Optional but recommended: log-transform heavy-tailed motif/count features
snap_features[feature_cols] = np.log1p(
    snap_features[feature_cols].clip(lower=0)
)

# ---------------------------------------------------------
# 4. Build full feature matrix aligned with graph node IDs
# ---------------------------------------------------------
# We create one row per graph node.
# Nodes missing from the Snap ML CSV receive zero features.

num_nodes = data.num_nodes

full_features = pd.DataFrame({
    "node_id": np.arange(num_nodes)
})

full_features = full_features.merge(
    snap_features[["node_id"] + feature_cols],
    on="node_id",
    how="left"
)

full_features[feature_cols] = (
    full_features[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

print("\nFull aligned feature matrix:")
print(full_features.shape)

missing_snap_rows = full_features[feature_cols].isna().all(axis=1).sum()
print("Nodes without Snap ML features:", missing_snap_rows)

# Standardize features
scaler = StandardScaler()

x_np = scaler.fit_transform(
    full_features[feature_cols].values
)

x = torch.tensor(
    x_np,
    dtype=torch.float
)

# Replace original node features with Snap ML features only
data.x = x

print("\nUpdated data.x shape:", data.x.shape)

# ---------------------------------------------------------
# 5. Create y vector
# ---------------------------------------------------------
# -1 = unlabeled
#  0 = normal
#  1 = fraud

y = torch.full(
    (num_nodes,),
    -1,
    dtype=torch.long
)

labelled_node_ids = labels_df["node_id"].values
label_values = labels_df["label"].values

y[labelled_node_ids] = torch.tensor(
    label_values,
    dtype=torch.long
)

data.y = y

print("\nLabel summary:")
print("Unlabelled nodes:", (data.y == -1).sum().item())
print("Normal labelled nodes:", (data.y == 0).sum().item())
print("Fraud labelled nodes:", (data.y == 1).sum().item())

# ---------------------------------------------------------
# 6. Train / validation / test split
# ---------------------------------------------------------

train_ids, temp_ids = train_test_split(
    labelled_node_ids,
    test_size=0.30,
    random_state=42,
    stratify=label_values
)

temp_labels = y[temp_ids].numpy()

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_ids] = True
val_mask[val_ids] = True
test_mask[test_ids] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print("\nSplit summary:")
print("Train nodes:", data.train_mask.sum().item())
print("Validation nodes:", data.val_mask.sum().item())
print("Test nodes:", data.test_mask.sum().item())

# ---------------------------------------------------------
# Select reusable GraphSAGE architecture
# ---------------------------------------------------------

GraphSAGE = GraphSAGEOneHidden

# ---------------------------------------------------------
# 8. Device, model, optimizer
# ---------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing device:", device)

data = data.to(device)

model = GraphSAGE(
    in_channels=data.x.shape[1],
    hidden_channels=128,
    out_channels=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

# ---------------------------------------------------------
# 9. Class weights
# ---------------------------------------------------------

train_labels = data.y[data.train_mask]

num_normal = (train_labels == 0).sum().item()
num_fraud = (train_labels == 1).sum().item()

class_weights = torch.tensor(
    [
        1.0 / num_normal,
        1.0 / num_fraud
    ],
    dtype=torch.float,
    device=device
)

class_weights = class_weights / class_weights.sum() * 2

print("\nClass weights:", class_weights)

# ---------------------------------------------------------
# 10. Training and evaluation functions
# ---------------------------------------------------------

def train_one_epoch():
    model.train()
    optimizer.zero_grad()

    out = model(data.x, data.edge_index)

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask],
        weight=class_weights
    )

    loss.backward()
    optimizer.step()

    return loss.item()


@torch.no_grad()
def evaluate(mask, threshold=0.5):
    model.eval()

    out = model(data.x, data.edge_index)

    logits = out[mask]
    labels = data.y[mask]

    probs = F.softmax(logits, dim=1)[:, 1]

    # Threshold-based fraud prediction
    preds = (probs >= threshold).long()

    acc = (preds == labels).float().mean().item()

    probs_np = probs.detach().cpu().numpy()
    preds_np = preds.detach().cpu().numpy()
    labels_np = labels.detach().cpu().numpy()

    roc_auc = roc_auc_score(labels_np, probs_np)
    pr_auc = average_precision_score(labels_np, probs_np)

    return acc, roc_auc, pr_auc, preds_np, probs_np, labels_np

# ---------------------------------------------------------
# 11. Train model
# ---------------------------------------------------------

EPOCHS = 150
best_val_pr_auc = 0.0

for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch()

    if epoch % 10 == 0:
        train_acc, train_roc, train_pr, _, _, _ = evaluate(data.train_mask)
        val_acc, val_roc, val_pr, _, _, _ = evaluate(data.val_mask)

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Train PR-AUC: {train_pr:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val ROC-AUC: {val_roc:.4f} | "
            f"Val PR-AUC: {val_pr:.4f}"
        )

        if val_pr > best_val_pr_auc:
            best_val_pr_auc = val_pr

            torch.save(
                model.state_dict(),
                BEST_MODEL_PATH
            )

            print("Saved best model.")

# ---------------------------------------------------------
# 12. Reload best model
# ---------------------------------------------------------

model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=device
    )
)

model.eval()

# ---------------------------------------------------------
# 13. Select fraud threshold on validation set
# ---------------------------------------------------------
# Instead of using default threshold = 0.5,
# choose the threshold that maximizes F1 on validation nodes.

_, _, _, _, val_probs, val_labels = evaluate(data.val_mask, threshold=0.5)

precision, recall, thresholds = precision_recall_curve(
    val_labels,
    val_probs
)

f1_scores = 2 * precision * recall / (precision + recall + 1e-12)

best_idx = np.argmax(f1_scores)

if best_idx == len(thresholds):
    best_threshold = 0.5
else:
    best_threshold = thresholds[best_idx]

print("\nBest validation threshold:", best_threshold)
print("Best validation F1:", f1_scores[best_idx])

# ---------------------------------------------------------
# 14. Final test evaluation
# ---------------------------------------------------------

test_acc, test_roc, test_pr, test_preds, test_probs, test_labels = evaluate(
    data.test_mask,
    threshold=best_threshold
)

print("\nFinal test results")
print("Test accuracy:", test_acc)
print("Test ROC-AUC:", test_roc)
print("Test PR-AUC:", test_pr)

print("\nClassification report:")
print(
    classification_report(
        test_labels,
        test_preds,
        target_names=["normal", "fraud"]
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(test_labels, test_preds))

# ---------------------------------------------------------
# 15. Predict fraud probability for every node
# ---------------------------------------------------------

with torch.no_grad():
    out = model(data.x, data.edge_index)
    probs = F.softmax(out, dim=1)
    fraud_probs = probs[:, 1].detach().cpu().numpy()

pred_df = pd.DataFrame({
    "node_id": np.arange(data.num_nodes),
    "fraud_probability": fraud_probs
})

pred_df["predicted_label"] = (
    pred_df["fraud_probability"] >= best_threshold
).astype(int)

node_mapping["node_id"] = node_mapping["node_id"].astype(int)

pred_df = pred_df.merge(
    node_mapping,
    on="node_id",
    how="left"
)

# Add known labels
y_cpu = data.y.detach().cpu().numpy()

pred_df["known_label"] = y_cpu
pred_df["is_labelled"] = pred_df["known_label"] != -1

pred_df = pred_df.sort_values(
    "fraud_probability",
    ascending=False
)

pred_df.to_csv(
    PRED_OUTPUT_PATH,
    index=False
)

print("\nSaved predictions:")
print(PRED_OUTPUT_PATH)

display(pred_df.head(20))

Loaded graph:
Data(x=[1828910, 20], edge_index=[2, 6092594], edge_attr=[6092594, 2], num_nodes=1828910)

Labels:
label
0    45000
1     4466
Name: count, dtype: int64

Snap ML feature file shape:
(1397958, 158)
   vertex_id  snap_motif_0_as_sender  snap_motif_1_as_sender  \
0   521656.0                     1.0                     0.0   
1   179411.0                     0.0                     0.0   
2   549884.0                     1.0                     0.0   
3   271831.0                     0.5                     0.0   
4   668516.0                     0.0                     0.0   

   snap_motif_2_as_sender  snap_motif_3_as_sender  snap_motif_4_as_sender  \
0                     0.0                     0.0                     0.0   
1                     0.0                     0.0                     0.0   
2                     0.0                     0.0                     0.0   
3                     0.0                     0.0                     0.0   
4                  

,node_id,fraud_probability,predicted_label,address,known_label,is_labelled
623257,623257,1.0,1,0x9a862994cf12c3a5f8a7572d73abaeb02f90b4a9,-1,False
504180,504180,1.0,1,0x71eb139c2d2484853e2d7388a4539338424ae485,-1,False
465239,465239,1.0,1,0xac646abb498a5aacc7745426d7cf275329b2c3a2,-1,False
642650,642650,1.0,1,0x997490514ef6da38c36c2d872703a4518ade90a0,-1,False
266540,266540,1.0,1,0x577333919b0f17db5a056c1b8bec0313ccc7fb22,-1,False
1015048,1015048,1.0,1,0xb199d6b0f291b70a062694f06f018ccc30e2a70b,-1,False
152582,152582,1.0,1,0xbdf510a33f988e13c7859ceb9f366b3d3992bcef,-1,False
809337,809337,1.0,1,0x1cddaf994a4518eab12ddb6264feff660adf54ae,-1,False
635010,635010,1.0,1,0x6d0f9b8cf8f5169f25e46179000019f04af5d7c1,-1,False
621839,621839,1.0,1,0x355faa07564058f4771a2c101744e32420acbafb,-1,False


### Result comment: Snap ML motif-only GraphSAGE with one hidden layer

The Snap ML motif-only model remains substantially weaker than models using the basic node features. It reaches ROC-AUC ≈ 0.887 and PR-AUC ≈ 0.428, with fraud precision around 0.40 and recall around 0.61. The confusion matrix still contains many false positives and false negatives. This seems to confirms the main motif-related conclusion: motif-derived features alone are not sufficient for robust fraud classification, probably because they do not capture the full structural and transactional profile of each account.


## 17. Snap ML motif features only: two hidden layers

Repeats the Snap ML motif-only experiment with a deeper two-hidden-layer GraphSAGE architecture.


In [ ]:
# =========================================================
# GraphSAGE semi-supervised node classification
# using ONLY Snap ML motif/node features
# 2 hidden layers
# =========================================================

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------

GRAPH_PATH = BASIC_GRAPH_PATH
LABEL_PATH = COMMUNITY_AWARE_LABELS_PATH
NODE_MAPPING_PATH = NODE_MAPPING_FILE_PATH

SNAP_FEATURE_PATH = SNAP_ML_FEATURE_PATH

BEST_MODEL_PATH = BEST_GRAPHSAGE_SNAP_ONLY_2H_PATH

PRED_OUTPUT_PATH = GRAPHSAGE_SNAP_ONLY_2H_PREDICTIONS_PATH

# ---------------------------------------------------------
# 2. Load graph, labels, node mapping, and Snap ML features
# ---------------------------------------------------------

data = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

labels_df = pd.read_csv(LABEL_PATH)
node_mapping = pd.read_csv(NODE_MAPPING_PATH)
snap_features = pd.read_csv(SNAP_FEATURE_PATH)

labels_df["node_id"] = labels_df["node_id"].astype(int)
labels_df["label"] = labels_df["label"].astype(int)

node_mapping["node_id"] = node_mapping["node_id"].astype(int)

print("Loaded graph:")
print(data)

print("\nLabels:")
print(labels_df["label"].value_counts())

print("\nSnap ML feature file shape:")
print(snap_features.shape)
print(snap_features.head())

# ---------------------------------------------------------
# 3. Prepare Snap ML node features
# ---------------------------------------------------------
# The Snap ML file uses vertex_id.
# We assume vertex_id corresponds to the same node_id used in the graph.

if "vertex_id" not in snap_features.columns:
    raise ValueError("Expected column 'vertex_id' in Snap ML feature file.")

snap_features = snap_features.rename(columns={"vertex_id": "node_id"})
snap_features["node_id"] = snap_features["node_id"].astype(int)

# Remove duplicated node rows if any
snap_features = snap_features.drop_duplicates(subset="node_id")

# Identify feature columns
id_cols = ["node_id", "address"]
feature_cols = [
    c for c in snap_features.columns
    if c not in id_cols
]

print("\nNumber of Snap ML feature columns:", len(feature_cols))

# Clean numeric values
for col in feature_cols:
    snap_features[col] = pd.to_numeric(
        snap_features[col],
        errors="coerce"
    )

snap_features[feature_cols] = (
    snap_features[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

# Optional but recommended: log-transform heavy-tailed motif/count features
snap_features[feature_cols] = np.log1p(
    snap_features[feature_cols].clip(lower=0)
)

# ---------------------------------------------------------
# 4. Build full feature matrix aligned with graph node IDs
# ---------------------------------------------------------
# We create one row per graph node.
# Nodes missing from the Snap ML CSV receive zero features.

num_nodes = data.num_nodes

full_features = pd.DataFrame({
    "node_id": np.arange(num_nodes)
})

full_features = full_features.merge(
    snap_features[["node_id"] + feature_cols],
    on="node_id",
    how="left"
)

full_features[feature_cols] = (
    full_features[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

print("\nFull aligned feature matrix:")
print(full_features.shape)

missing_snap_rows = full_features[feature_cols].isna().all(axis=1).sum()
print("Nodes without Snap ML features:", missing_snap_rows)

# Standardize features
scaler = StandardScaler()

x_np = scaler.fit_transform(
    full_features[feature_cols].values
)

x = torch.tensor(
    x_np,
    dtype=torch.float
)

# Replace original node features with Snap ML features only
data.x = x

print("\nUpdated data.x shape:", data.x.shape)

# ---------------------------------------------------------
# 5. Create y vector
# ---------------------------------------------------------
# -1 = unlabeled
#  0 = normal
#  1 = fraud

y = torch.full(
    (num_nodes,),
    -1,
    dtype=torch.long
)

labelled_node_ids = labels_df["node_id"].values
label_values = labels_df["label"].values

y[labelled_node_ids] = torch.tensor(
    label_values,
    dtype=torch.long
)

data.y = y

print("\nLabel summary:")
print("Unlabelled nodes:", (data.y == -1).sum().item())
print("Normal labelled nodes:", (data.y == 0).sum().item())
print("Fraud labelled nodes:", (data.y == 1).sum().item())

# ---------------------------------------------------------
# 6. Train / validation / test split
# ---------------------------------------------------------

train_ids, temp_ids = train_test_split(
    labelled_node_ids,
    test_size=0.30,
    random_state=42,
    stratify=label_values
)

temp_labels = y[temp_ids].numpy()

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_ids] = True
val_mask[val_ids] = True
test_mask[test_ids] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print("\nSplit summary:")
print("Train nodes:", data.train_mask.sum().item())
print("Validation nodes:", data.val_mask.sum().item())
print("Test nodes:", data.test_mask.sum().item())

# ---------------------------------------------------------
# Select reusable GraphSAGE architecture
# ---------------------------------------------------------

GraphSAGE = GraphSAGETwoHidden

# ---------------------------------------------------------
# 8. Device, model, optimizer
# ---------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing device:", device)

data = data.to(device)

model = GraphSAGE(
    in_channels=data.x.shape[1],
    hidden_channels=128,
    out_channels=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

# ---------------------------------------------------------
# 9. Class weights
# ---------------------------------------------------------

train_labels = data.y[data.train_mask]

num_normal = (train_labels == 0).sum().item()
num_fraud = (train_labels == 1).sum().item()

class_weights = torch.tensor(
    [
        1.0 / num_normal,
        1.0 / num_fraud
    ],
    dtype=torch.float,
    device=device
)

class_weights = class_weights / class_weights.sum() * 2

print("\nClass weights:", class_weights)

# ---------------------------------------------------------
# Reuse standard training and evaluation functions
# ---------------------------------------------------------

# train_one_epoch() and evaluate() are defined in the shared utilities section.

# ---------------------------------------------------------
# 11. Train model
# ---------------------------------------------------------

EPOCHS = 150
best_val_pr_auc = 0.0

for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch()

    if epoch % 10 == 0:
        train_acc, train_roc, train_pr, _, _, _ = evaluate(data.train_mask)
        val_acc, val_roc, val_pr, _, _, _ = evaluate(data.val_mask)

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Train PR-AUC: {train_pr:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val ROC-AUC: {val_roc:.4f} | "
            f"Val PR-AUC: {val_pr:.4f}"
        )

        if val_pr > best_val_pr_auc:
            best_val_pr_auc = val_pr

            torch.save(
                model.state_dict(),
                BEST_MODEL_PATH
            )

            print("Saved best model.")

# ---------------------------------------------------------
# 12. Reload best model
# ---------------------------------------------------------

model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=device
    )
)

model.eval()

# ---------------------------------------------------------
# 13. Select fraud threshold on validation set
# ---------------------------------------------------------
# Instead of using default threshold = 0.5,
# choose the threshold that maximizes F1 on validation nodes.

_, _, _, _, val_probs, val_labels = evaluate(data.val_mask, threshold=0.5)

precision, recall, thresholds = precision_recall_curve(
    val_labels,
    val_probs
)

f1_scores = 2 * precision * recall / (precision + recall + 1e-12)

best_idx = np.argmax(f1_scores)

if best_idx == len(thresholds):
    best_threshold = 0.5
else:
    best_threshold = thresholds[best_idx]

print("\nBest validation threshold:", best_threshold)
print("Best validation F1:", f1_scores[best_idx])

# ---------------------------------------------------------
# 14. Final test evaluation
# ---------------------------------------------------------

test_acc, test_roc, test_pr, test_preds, test_probs, test_labels = evaluate(
    data.test_mask,
    threshold=best_threshold
)

print("\nFinal test results")
print("Test accuracy:", test_acc)
print("Test ROC-AUC:", test_roc)
print("Test PR-AUC:", test_pr)

print("\nClassification report:")
print(
    classification_report(
        test_labels,
        test_preds,
        target_names=["normal", "fraud"]
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(test_labels, test_preds))

# ---------------------------------------------------------
# 15. Predict fraud probability for every node
# ---------------------------------------------------------

with torch.no_grad():
    out = model(data.x, data.edge_index)
    probs = F.softmax(out, dim=1)
    fraud_probs = probs[:, 1].detach().cpu().numpy()

pred_df = pd.DataFrame({
    "node_id": np.arange(data.num_nodes),
    "fraud_probability": fraud_probs
})

pred_df["predicted_label"] = (
    pred_df["fraud_probability"] >= best_threshold
).astype(int)

node_mapping["node_id"] = node_mapping["node_id"].astype(int)

pred_df = pred_df.merge(
    node_mapping,
    on="node_id",
    how="left"
)

# Add known labels
y_cpu = data.y.detach().cpu().numpy()

pred_df["known_label"] = y_cpu
pred_df["is_labelled"] = pred_df["known_label"] != -1

pred_df = pred_df.sort_values(
    "fraud_probability",
    ascending=False
)

pred_df.to_csv(
    PRED_OUTPUT_PATH,
    index=False
)

print("\nSaved predictions:")
print(PRED_OUTPUT_PATH)

display(pred_df.head(20))

Loaded graph:
Data(x=[1828910, 20], edge_index=[2, 6092594], edge_attr=[6092594, 2], num_nodes=1828910)

Labels:
label
0    45000
1     4466
Name: count, dtype: int64

Snap ML feature file shape:
(1397958, 158)
   vertex_id  snap_motif_0_as_sender  snap_motif_1_as_sender  \
0   521656.0                     1.0                     0.0   
1   179411.0                     0.0                     0.0   
2   549884.0                     1.0                     0.0   
3   271831.0                     0.5                     0.0   
4   668516.0                     0.0                     0.0   

   snap_motif_2_as_sender  snap_motif_3_as_sender  snap_motif_4_as_sender  \
0                     0.0                     0.0                     0.0   
1                     0.0                     0.0                     0.0   
2                     0.0                     0.0                     0.0   
3                     0.0                     0.0                     0.0   
4                  

,node_id,fraud_probability,predicted_label,address,known_label,is_labelled
116051,116051,1.0,1,0x77021e9b01d3936bc84692c55273bcb9f14d7989,-1,False
403928,403928,1.0,1,0x0fd45e74ba0d483aeca2d3d0e415c4a34d5d9421,-1,False
347790,347790,1.0,1,0x0491147275f5d6cd716bea8813a393d489cb7685,-1,False
794811,794811,1.0,1,0x37968745b764c2afe157c076a7cad209c453f0a6,-1,False
452911,452911,1.0,1,0x7a2cb4e0d5b25311018fcdcf26e8a111096e091b,-1,False
1326487,1326487,1.0,1,0x7560c9c5f693c37e5e8420c9bcd5934a9ef2e2d1,-1,False
1681128,1681128,1.0,1,0x0fd209a115576637e179f2d36e8b045af5ba7d2d,-1,False
1645614,1645614,1.0,1,0xab422a9b3767f4f1a2443882f5c0d1a01f30cde2,-1,False
1228967,1228967,1.0,1,0x0b1aa433d830c6ebd674979c1c618498578717a0,-1,False
312852,312852,1.0,1,0xf97261bddb72b11216bbf829fb8907ec14d61c66,-1,False


### Result comment: Snap ML motif-only GraphSAGE with two hidden layers

The two-hidden-layer Snap ML motif-only model improves slightly over the one-hidden-layer version, but it remains weak overall. PR-AUC increases from about 0.428 to about 0.491, and recall increases from about 0.61 to about 0.69, while fraud precision stays around 0.40. The added depth helps the model extract more signal from the Snap motif table, but the motif-only representation is still clearly insufficient compared with models that include basic node features or GraphSAGE embeddings.


## 18. GraphSAGE + temporal motif late-fusion MLP

Uses GraphSAGE as an encoder over the basic node features, concatenates the learned node embeddings with temporal motif counts, and classifies nodes with an MLP.


In [5]:
# =========================================================
# GraphSAGE + Temporal Motif Late-Fusion MLP
# =========================================================

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------

GRAPH_PATH = BASIC_GRAPH_PATH
NODE_FEATURES_PATH = NODE_BASIC_FEATURES_PATH
NODE_MAPPING_PATH = NODE_MAPPING_FILE_PATH
LABEL_PATH = COMMUNITY_AWARE_LABELS_PATH
MOTIF_PATH = TEMPORAL_MOTIF_COUNTS_PATH

BEST_MODEL_PATH = BEST_GRAPHSAGE_TEMPORAL_FUSION_PATH
PRED_OUTPUT_PATH = GRAPHSAGE_TEMPORAL_FUSION_PREDICTIONS_PATH

# ---------------------------------------------------------
# 2. Load graph, features, labels, motifs
# ---------------------------------------------------------

data = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

node_features = pd.read_csv(NODE_FEATURES_PATH)
node_mapping = pd.read_csv(NODE_MAPPING_PATH)
labels_df = pd.read_csv(LABEL_PATH)
motifs = pd.read_csv(MOTIF_PATH)

if "node_id" not in node_features.columns:
    node_features["node_id"] = node_features.index

node_features["node_id"] = node_features["node_id"].astype(int)
node_mapping["node_id"] = node_mapping["node_id"].astype(int)
labels_df["node_id"] = labels_df["node_id"].astype(int)
labels_df["label"] = labels_df["label"].astype(int)

node_mapping["address"] = (
    node_mapping["address"]
    .astype(str)
    .str.lower()
    .str.strip()
)

motifs["address"] = (
    motifs["address"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Attach node_id to motifs
motifs = motifs.merge(
    node_mapping[["address", "node_id"]],
    on="address",
    how="inner"
)

print(data)
print("Node features:", node_features.shape)
print("Motifs matched to graph:", motifs.shape)
print("Labels:")
print(labels_df["label"].value_counts())

# ---------------------------------------------------------
# 3. Build basic feature matrix
# ---------------------------------------------------------

node_features = node_features.sort_values("node_id").reset_index(drop=True)

assert node_features["node_id"].iloc[0] == 0
assert node_features["node_id"].iloc[-1] == data.num_nodes - 1
assert len(node_features) == data.num_nodes

exclude_cols = ["node_id", "address"]

basic_feature_cols = [
    col for col in node_features.columns
    if col not in exclude_cols
]

x_basic_np = node_features[basic_feature_cols].replace(
    [np.inf, -np.inf],
    0.0
).fillna(0.0).values

print("Basic feature columns:", len(basic_feature_cols))

# ---------------------------------------------------------
# 4. Build motif feature matrix
# ---------------------------------------------------------

motif_cols = [
    "temporal_out_star",
    "temporal_in_star",
    "temporal_chain",
    "temporal_reciprocal",
    "temporal_cycle_3"
]

motif_cols = [col for col in motif_cols if col in motifs.columns]

motif_features = pd.DataFrame({
    "node_id": np.arange(data.num_nodes)
})

motif_features = motif_features.merge(
    motifs[["node_id"] + motif_cols],
    on="node_id",
    how="left"
)

for col in motif_cols:
    motif_features[col] = (
        motif_features[col]
        .replace([np.inf, -np.inf], 0.0)
        .fillna(0.0)
    )

# Standardize motif features because they may have different scales
scaler = StandardScaler()
x_motif_np = scaler.fit_transform(motif_features[motif_cols].values)

print("Motif feature columns:", motif_cols)

# ---------------------------------------------------------
# 5. Store separate matrices in data object
# ---------------------------------------------------------

data.x_basic = torch.tensor(
    x_basic_np,
    dtype=torch.float
)

data.x_motif = torch.tensor(
    x_motif_np,
    dtype=torch.float
)

print("x_basic shape:", data.x_basic.shape)
print("x_motif shape:", data.x_motif.shape)

# ---------------------------------------------------------
# 6. Create labels and masks
# ---------------------------------------------------------

num_nodes = data.num_nodes

y = torch.full(
    (num_nodes,),
    -1,
    dtype=torch.long
)

labelled_node_ids = labels_df["node_id"].values
label_values = labels_df["label"].values

y[labelled_node_ids] = torch.tensor(
    label_values,
    dtype=torch.long
)

data.y = y

train_ids, temp_ids = train_test_split(
    labelled_node_ids,
    test_size=0.30,
    random_state=42,
    stratify=label_values
)

temp_labels = y[temp_ids].numpy()

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_ids] = True
val_mask[val_ids] = True
test_mask[test_ids] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print("Train nodes:", data.train_mask.sum().item())
print("Validation nodes:", data.val_mask.sum().item())
print("Test nodes:", data.test_mask.sum().item())

# ---------------------------------------------------------
# 7. Define late-fusion model
# ---------------------------------------------------------

class GraphSAGEWithMotifMLP(torch.nn.Module):
    def __init__(
        self,
        basic_in_channels,
        motif_in_channels,
        sage_hidden_channels=64,
        embedding_channels=64,
        mlp_hidden_channels=64,
        out_channels=2,
        dropout=0.3
    ):
        super().__init__()

        self.conv1 = SAGEConv(
            basic_in_channels,
            sage_hidden_channels
        )

        self.conv2 = SAGEConv(
            sage_hidden_channels,
            embedding_channels
        )

        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(
                embedding_channels + motif_in_channels,
                mlp_hidden_channels
            ),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(
                mlp_hidden_channels,
                out_channels
            )
        )

        self.dropout = dropout

    def forward(self, x_basic, x_motif, edge_index):
        h = self.conv1(x_basic, edge_index)
        h = F.relu(h)
        h = F.dropout(
            h,
            p=self.dropout,
            training=self.training
        )

        h = self.conv2(h, edge_index)
        h = F.relu(h)

        z = torch.cat(
            [h, x_motif],
            dim=1
        )

        out = self.mlp(z)

        return out

# ---------------------------------------------------------
# 8. Device, model, optimizer
# ---------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

data = data.to(device)

model = GraphSAGEWithMotifMLP(
    basic_in_channels=data.x_basic.shape[1],
    motif_in_channels=data.x_motif.shape[1],
    sage_hidden_channels=64,
    embedding_channels=64,
    mlp_hidden_channels=64,
    out_channels=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

# ---------------------------------------------------------
# 9. Class weights
# ---------------------------------------------------------

train_labels = data.y[data.train_mask]

num_normal = (train_labels == 0).sum().item()
num_fraud = (train_labels == 1).sum().item()

class_weights = torch.tensor(
    [
        1.0 / num_normal,
        1.0 / num_fraud
    ],
    dtype=torch.float,
    device=device
)

class_weights = class_weights / class_weights.sum() * 2

print("Class weights:", class_weights)

# ---------------------------------------------------------
# 10. Train / evaluate functions
# ---------------------------------------------------------

def train_one_epoch():
    model.train()
    optimizer.zero_grad()

    out = model(
        data.x_basic,
        data.x_motif,
        data.edge_index
    )

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask],
        weight=class_weights
    )

    loss.backward()
    optimizer.step()

    return loss.item()


@torch.no_grad()
def evaluate(mask):
    model.eval()

    out = model(
        data.x_basic,
        data.x_motif,
        data.edge_index
    )

    logits = out[mask]
    labels = data.y[mask]

    probs = F.softmax(logits, dim=1)[:, 1]
    preds_argmax = logits.argmax(dim=1)

    acc = (preds_argmax == labels).float().mean().item()

    probs_np = probs.detach().cpu().numpy()
    preds_np = preds_argmax.detach().cpu().numpy()
    labels_np = labels.detach().cpu().numpy()

    roc_auc = roc_auc_score(labels_np, probs_np)
    pr_auc = average_precision_score(labels_np, probs_np)

    return acc, roc_auc, pr_auc, preds_np, probs_np, labels_np

# ---------------------------------------------------------
# 11. Train model
# ---------------------------------------------------------

EPOCHS = 150

best_val_pr_auc = 0

for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch()

    if epoch % 10 == 0:
        train_acc, train_roc, train_pr, _, _, _ = evaluate(data.train_mask)
        val_acc, val_roc, val_pr, _, _, _ = evaluate(data.val_mask)

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Train PR-AUC: {train_pr:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val ROC-AUC: {val_roc:.4f} | "
            f"Val PR-AUC: {val_pr:.4f}"
        )

        if val_pr > best_val_pr_auc:
            best_val_pr_auc = val_pr

            torch.save(
                model.state_dict(),
                BEST_MODEL_PATH
            )

            print("Saved best model.")

# ---------------------------------------------------------
# 12. Load best model
# ---------------------------------------------------------

model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=device
    )
)

# ---------------------------------------------------------
# 13. Select validation threshold maximizing F1
# ---------------------------------------------------------

val_acc, val_roc, val_pr, val_preds_argmax, val_probs, val_labels = evaluate(
    data.val_mask
)

thresholds = np.arange(0.05, 0.96, 0.01)

threshold_results = []

for threshold in thresholds:
    val_preds_thresholded = (val_probs >= threshold).astype(int)

    precision = precision_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    recall = recall_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    f1 = f1_score(
        val_labels,
        val_preds_thresholded,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results = pd.DataFrame(threshold_results)

best_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold = float(best_row["threshold"])

print("\nBest threshold selected on validation set:")
print(best_row)

threshold_results.to_csv(
    GRAPHSAGE_TEMPORAL_FUSION_THRESHOLD_TABLE_PATH,
    index=False
)

# ---------------------------------------------------------
# 14. Final test evaluation
# ---------------------------------------------------------

test_acc_argmax, test_roc, test_pr, test_preds_argmax, test_probs, test_labels = evaluate(
    data.test_mask
)

test_preds_thresholded = (test_probs >= best_threshold).astype(int)

print("\nFinal test results with validation-selected threshold")
print("Selected threshold:", best_threshold)
print("Test ROC-AUC:", test_roc)
print("Test PR-AUC:", test_pr)

print("\nClassification report:")
print(
    classification_report(
        test_labels,
        test_preds_thresholded,
        target_names=["normal", "fraud"],
        zero_division=0
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(test_labels, test_preds_thresholded))

# ---------------------------------------------------------
# 15. Predict all nodes
# ---------------------------------------------------------

model.eval()

with torch.no_grad():
    out = model(
        data.x_basic,
        data.x_motif,
        data.edge_index
    )

    probs = F.softmax(out, dim=1)
    fraud_probs = probs[:, 1].detach().cpu().numpy()

pred_df = pd.DataFrame({
    "node_id": np.arange(data.num_nodes),
    "fraud_probability": fraud_probs
})

node_mapping["node_id"] = node_mapping["node_id"].astype(int)

pred_df = pred_df.merge(
    node_mapping,
    on="node_id",
    how="left"
)

y_cpu = data.y.detach().cpu().numpy()

pred_df["known_label"] = y_cpu
pred_df["is_labelled"] = pred_df["known_label"] != -1

pred_df["predicted_label_thresholded"] = (
    pred_df["fraud_probability"] >= best_threshold
).astype(int)

pred_df["predicted_label_name"] = pred_df[
    "predicted_label_thresholded"
].map({
    0: "normal",
    1: "fraud"
})

pred_df = pred_df.sort_values(
    "fraud_probability",
    ascending=False
)

pred_df.to_csv(
    PRED_OUTPUT_PATH,
    index=False
)

print("\nSaved predictions:")
print(PRED_OUTPUT_PATH)

display(pred_df.head(20))

Data(x=[1828910, 20], edge_index=[2, 6092594], edge_attr=[6092594, 2], num_nodes=1828910)
Node features: (1828910, 21)
Motifs matched to graph: (1828910, 7)
Labels:
label
0    45000
1     4466
Name: count, dtype: int64
Basic feature columns: 20
Motif feature columns: ['temporal_out_star', 'temporal_in_star', 'temporal_chain', 'temporal_reciprocal', 'temporal_cycle_3']
x_basic shape: torch.Size([1828910, 20])
x_motif shape: torch.Size([1828910, 5])
Train nodes: 34626
Validation nodes: 7420
Test nodes: 7420
Using device: cuda
Class weights: tensor([0.1806, 1.8194], device='cuda:0')
Epoch 010 | Loss: 0.5732 | Train Acc: 0.5643 | Train PR-AUC: 0.4723 | Val Acc: 0.5698 | Val ROC-AUC: 0.9212 | Val PR-AUC: 0.4388
Saved best model.
Epoch 020 | Loss: 0.4326 | Train Acc: 0.8579 | Train PR-AUC: 0.5652 | Val Acc: 0.8539 | Val ROC-AUC: 0.9213 | Val PR-AUC: 0.5272
Saved best model.
Epoch 030 | Loss: 0.3569 | Train Acc: 0.8255 | Train PR-AUC: 0.6455 | Val Acc: 0.8244 | Val ROC-AUC: 0.9389 | Val PR-AU

,node_id,fraud_probability,address,known_label,is_labelled,predicted_label_thresholded,predicted_label_name
1806,1806,1.000000,0xa26148ae51fa8e787df319c04137602cc018b521,1,True,1,fraud
46,46,1.000000,0x1ab4973a48dc892cd9971ece8e01dcc7688f8f23,1,True,1,fraud
516,516,1.000000,0x458c2a06ba8dc3def078513ad98e9da8ced39e02,1,True,1,fraud
80,80,1.000000,0xa1abfa21f80ecf401bd41365adbb6fef6fefdf09,1,True,1,fraud
980,980,1.000000,0x974caa59e49682cda0ad2bbe82983419a2ecc400,1,True,1,fraud
27,27,1.000000,0x28c6c06298d514db089934071355e5743bf21d60,1,True,1,fraud
512,512,1.000000,0xf30ba13e4b04ce5dc4d254ae5fa95477800f0eb0,-1,False,1,fraud
31,31,1.000000,0x264bd8291fae1d75db2c5f573b07faa6715997b5,-1,False,1,fraud
798,798,1.000000,0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f,-1,False,1,fraud
403,403,1.000000,0x2cff890f0378a11913b6129b2e97417a2c302680,-1,False,1,fraud


### Result comment: GraphSAGE embeddings plus temporal motif late-fusion MLP

The temporal motif late-fusion model performs strongly and is the most convincing architecture for using motifs. It reaches ROC-AUC ≈ 0.982 and PR-AUC ≈ 0.869, with fraud precision around 0.79 and recall around 0.86. Compared with direct motif input to GraphSAGE, late fusion achieves a better use of temporal motifs: GraphSAGE first learns topology-aware embeddings from the basic features and graph structure, then the MLP combines those embeddings with explicit motif counts. This keeps temporal motifs as behavioural node-level signals instead of diffusing them through message passing.


## 19. GraphSAGE encoder + Snap ML motif features + MLP

Uses a two-layer GraphSAGE encoder on the original basic features, concatenates the resulting embeddings with Snap ML motif-derived features, and trains an MLP classifier on the labeled nodes.


In [ ]:
# =========================================================
# GraphSAGE encoder + Snap ML motif features + MLP classifier
# Semi-supervised node classification
# =========================================================

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------

GRAPH_PATH = BASIC_GRAPH_PATH
LABEL_PATH = COMMUNITY_AWARE_LABELS_PATH
NODE_MAPPING_PATH = NODE_MAPPING_FILE_PATH
NODE_FEATURES_PATH = NODE_BASIC_FEATURES_PATH

SNAP_FEATURE_PATH = SNAP_ML_FEATURE_PATH

BEST_MODEL_PATH = BEST_GRAPHSAGE_SNAP_FUSION_PATH

PRED_OUTPUT_PATH = GRAPHSAGE_SNAP_FUSION_PREDICTIONS_PATH

# ---------------------------------------------------------
# 2. Load files
# ---------------------------------------------------------

data = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

labels_df = pd.read_csv(LABEL_PATH)
node_mapping = pd.read_csv(NODE_MAPPING_PATH)
basic_features = pd.read_csv(NODE_FEATURES_PATH)
snap_features = pd.read_csv(SNAP_FEATURE_PATH)

labels_df["node_id"] = labels_df["node_id"].astype(int)
labels_df["label"] = labels_df["label"].astype(int)
node_mapping["node_id"] = node_mapping["node_id"].astype(int)

print("Graph:")
print(data)

print("\nLabels:")
print(labels_df["label"].value_counts())

print("\nBasic feature table:", basic_features.shape)
print("Snap ML feature table:", snap_features.shape)

# ---------------------------------------------------------
# 3. Prepare basic node features
# ---------------------------------------------------------
# These are the original engineered node features.
# They will be used ONLY by the GraphSAGE encoder.

if "node_id" not in basic_features.columns:
    basic_features["node_id"] = basic_features.index

basic_features["node_id"] = basic_features["node_id"].astype(int)

basic_id_cols = ["node_id", "address"]
basic_feature_cols = [
    c for c in basic_features.columns
    if c not in basic_id_cols
]

for col in basic_feature_cols:
    basic_features[col] = pd.to_numeric(
        basic_features[col],
        errors="coerce"
    )

basic_features[basic_feature_cols] = (
    basic_features[basic_feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

# ---------------------------------------------------------
# 4. Prepare Snap ML motif-derived features
# ---------------------------------------------------------
# These features will NOT go through GraphSAGE.
# They will be concatenated with the GraphSAGE embeddings before the MLP.

if "vertex_id" not in snap_features.columns:
    raise ValueError("Expected column 'vertex_id' in Snap ML feature file.")

snap_features = snap_features.rename(columns={"vertex_id": "node_id"})
snap_features["node_id"] = snap_features["node_id"].astype(int)
snap_features = snap_features.drop_duplicates(subset="node_id")

motif_id_cols = ["node_id", "address"]
motif_feature_cols = [
    c for c in snap_features.columns
    if c not in motif_id_cols
]

for col in motif_feature_cols:
    snap_features[col] = pd.to_numeric(
        snap_features[col],
        errors="coerce"
    )

snap_features[motif_feature_cols] = (
    snap_features[motif_feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

# Log-transform motif features because counts are usually heavy-tailed
snap_features[motif_feature_cols] = np.log1p(
    snap_features[motif_feature_cols].clip(lower=0)
)

# ---------------------------------------------------------
# 5. Align both feature matrices to graph node IDs
# ---------------------------------------------------------

num_nodes = data.num_nodes

all_nodes_df = pd.DataFrame({
    "node_id": np.arange(num_nodes)
})

basic_aligned = all_nodes_df.merge(
    basic_features[["node_id"] + basic_feature_cols],
    on="node_id",
    how="left"
)

motif_aligned = all_nodes_df.merge(
    snap_features[["node_id"] + motif_feature_cols],
    on="node_id",
    how="left"
)

basic_aligned[basic_feature_cols] = (
    basic_aligned[basic_feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

motif_aligned[motif_feature_cols] = (
    motif_aligned[motif_feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

print("\nAligned basic features:", basic_aligned.shape)
print("Aligned motif features:", motif_aligned.shape)

# ---------------------------------------------------------
# 6. Standardize basic and motif features separately
# ---------------------------------------------------------

basic_scaler = StandardScaler()
motif_scaler = StandardScaler()

x_basic_np = basic_scaler.fit_transform(
    basic_aligned[basic_feature_cols].values
)

x_motif_np = motif_scaler.fit_transform(
    motif_aligned[motif_feature_cols].values
)

x_basic = torch.tensor(
    x_basic_np,
    dtype=torch.float
)

x_motif = torch.tensor(
    x_motif_np,
    dtype=torch.float
)

# Store as separate tensors
data.x_basic = x_basic
data.x_motif = x_motif

print("\nx_basic shape:", data.x_basic.shape)
print("x_motif shape:", data.x_motif.shape)

# ---------------------------------------------------------
# 7. Create labels
# ---------------------------------------------------------
# -1 = unlabeled
#  0 = normal
#  1 = fraud

y = torch.full(
    (num_nodes,),
    -1,
    dtype=torch.long
)

labelled_node_ids = labels_df["node_id"].values
label_values = labels_df["label"].values

y[labelled_node_ids] = torch.tensor(
    label_values,
    dtype=torch.long
)

data.y = y

print("\nLabel summary:")
print("Unlabelled:", (data.y == -1).sum().item())
print("Normal:", (data.y == 0).sum().item())
print("Fraud:", (data.y == 1).sum().item())

# ---------------------------------------------------------
# 8. Train / validation / test split
# ---------------------------------------------------------

train_ids, temp_ids = train_test_split(
    labelled_node_ids,
    test_size=0.30,
    random_state=42,
    stratify=label_values
)

temp_labels = y[temp_ids].numpy()

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_ids] = True
val_mask[val_ids] = True
test_mask[test_ids] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print("\nSplit summary:")
print("Train:", data.train_mask.sum().item())
print("Validation:", data.val_mask.sum().item())
print("Test:", data.test_mask.sum().item())

# ---------------------------------------------------------
# 9. Define GraphSAGE encoder + motif MLP model
# ---------------------------------------------------------

class GraphSAGEEncoderMotifMLP(torch.nn.Module):
    def __init__(
        self,
        basic_in_channels,
        motif_in_channels,
        hidden_channels=64,
        embedding_dim=64,
        mlp_hidden=128,
        out_channels=2,
        dropout=0.3
    ):
        super().__init__()

        # 2-hidden-layer GraphSAGE encoder on basic features
        self.sage1 = SAGEConv(basic_in_channels, hidden_channels)
        self.sage2 = SAGEConv(hidden_channels, hidden_channels)
        self.sage3 = SAGEConv(hidden_channels, embedding_dim)

        self.dropout = dropout

        # MLP classifier on [GraphSAGE embedding + motif features]
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(embedding_dim + motif_in_channels, mlp_hidden),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(mlp_hidden, out_channels)
        )

    def forward(self, x_basic, x_motif, edge_index):
        z = self.sage1(x_basic, edge_index)
        z = F.relu(z)
        z = F.dropout(z, p=self.dropout, training=self.training)

        z = self.sage2(z, edge_index)
        z = F.relu(z)
        z = F.dropout(z, p=self.dropout, training=self.training)

        z = self.sage3(z, edge_index)

        combined = torch.cat([z, x_motif], dim=1)

        out = self.mlp(combined)

        return out

# ---------------------------------------------------------
# 10. Device, model, optimizer
# ---------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing device:", device)

data = data.to(device)

model = GraphSAGEEncoderMotifMLP(
    basic_in_channels=data.x_basic.shape[1],
    motif_in_channels=data.x_motif.shape[1],
    hidden_channels=64,
    embedding_dim=64,
    mlp_hidden=64,
    out_channels=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

# ---------------------------------------------------------
# 11. Class weights
# ---------------------------------------------------------

train_labels = data.y[data.train_mask]

num_normal = (train_labels == 0).sum().item()
num_fraud = (train_labels == 1).sum().item()

class_weights = torch.tensor(
    [
        1.0 / num_normal,
        1.0 / num_fraud
    ],
    dtype=torch.float,
    device=device
)

class_weights = class_weights / class_weights.sum() * 2

print("\nClass weights:", class_weights)

# ---------------------------------------------------------
# 12. Training and evaluation functions
# ---------------------------------------------------------

def train_one_epoch():
    model.train()
    optimizer.zero_grad()

    out = model(
        data.x_basic,
        data.x_motif,
        data.edge_index
    )

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask],
        weight=class_weights
    )

    loss.backward()
    optimizer.step()

    return loss.item()


@torch.no_grad()
def evaluate(mask, threshold=0.5):
    model.eval()

    out = model(
        data.x_basic,
        data.x_motif,
        data.edge_index
    )

    logits = out[mask]
    labels = data.y[mask]

    probs = F.softmax(logits, dim=1)[:, 1]
    preds = (probs >= threshold).long()

    acc = (preds == labels).float().mean().item()

    probs_np = probs.detach().cpu().numpy()
    preds_np = preds.detach().cpu().numpy()
    labels_np = labels.detach().cpu().numpy()

    roc_auc = roc_auc_score(labels_np, probs_np)
    pr_auc = average_precision_score(labels_np, probs_np)

    return acc, roc_auc, pr_auc, preds_np, probs_np, labels_np

# ---------------------------------------------------------
# 13. Train model
# ---------------------------------------------------------

EPOCHS = 150
best_val_pr_auc = 0.0

for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch()

    if epoch % 10 == 0:
        train_acc, train_roc, train_pr, _, _, _ = evaluate(data.train_mask)
        val_acc, val_roc, val_pr, _, _, _ = evaluate(data.val_mask)

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Train PR-AUC: {train_pr:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val ROC-AUC: {val_roc:.4f} | "
            f"Val PR-AUC: {val_pr:.4f}"
        )

        if val_pr > best_val_pr_auc:
            best_val_pr_auc = val_pr

            torch.save(
                model.state_dict(),
                BEST_MODEL_PATH
            )

            print("Saved best model.")

# ---------------------------------------------------------
# 14. Reload best model
# ---------------------------------------------------------

model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=device
    )
)

model.eval()

# ---------------------------------------------------------
# 15. Select threshold on validation set
# ---------------------------------------------------------

_, _, _, _, val_probs, val_labels = evaluate(
    data.val_mask,
    threshold=0.5
)

precision, recall, thresholds = precision_recall_curve(
    val_labels,
    val_probs
)

f1_scores = 2 * precision * recall / (
    precision + recall + 1e-12
)

best_idx = np.argmax(f1_scores)

if best_idx == len(thresholds):
    best_threshold = 0.5
else:
    best_threshold = thresholds[best_idx]

print("\nBest validation threshold:", best_threshold)
print("Best validation F1:", f1_scores[best_idx])

# ---------------------------------------------------------
# 16. Final test evaluation
# ---------------------------------------------------------

test_acc, test_roc, test_pr, test_preds, test_probs, test_labels = evaluate(
    data.test_mask,
    threshold=best_threshold
)

print("\nFinal test results")
print("Test accuracy:", test_acc)
print("Test ROC-AUC:", test_roc)
print("Test PR-AUC:", test_pr)

print("\nClassification report:")
print(
    classification_report(
        test_labels,
        test_preds,
        target_names=["normal", "fraud"]
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(test_labels, test_preds))

# ---------------------------------------------------------
# 17. Predict fraud probability for every node
# ---------------------------------------------------------

with torch.no_grad():
    out = model(
        data.x_basic,
        data.x_motif,
        data.edge_index
    )

    probs = F.softmax(out, dim=1)
    fraud_probs = probs[:, 1].detach().cpu().numpy()

pred_df = pd.DataFrame({
    "node_id": np.arange(data.num_nodes),
    "fraud_probability": fraud_probs
})

pred_df["predicted_label"] = (
    pred_df["fraud_probability"] >= best_threshold
).astype(int)

node_mapping["node_id"] = node_mapping["node_id"].astype(int)

pred_df = pred_df.merge(
    node_mapping,
    on="node_id",
    how="left"
)

y_cpu = data.y.detach().cpu().numpy()

pred_df["known_label"] = y_cpu
pred_df["is_labelled"] = pred_df["known_label"] != -1

pred_df = pred_df.sort_values(
    "fraud_probability",
    ascending=False
)

pred_df.to_csv(
    PRED_OUTPUT_PATH,
    index=False
)

print("\nSaved predictions:")
print(PRED_OUTPUT_PATH)

display(pred_df.head(20))

Graph:
Data(x=[1828910, 20], edge_index=[2, 6092594], edge_attr=[6092594, 2], num_nodes=1828910)

Labels:
label
0    45000
1     4466
Name: count, dtype: int64

Basic feature table: (1828910, 21)
Snap ML feature table: (1397958, 158)

Aligned basic features: (1828910, 21)
Aligned motif features: (1828910, 158)

x_basic shape: torch.Size([1828910, 20])
x_motif shape: torch.Size([1828910, 157])

Label summary:
Unlabelled: 1779444
Normal: 45000
Fraud: 4466

Split summary:
Train: 34626
Validation: 7420
Test: 7420

Using device: cuda

Class weights: tensor([0.1806, 1.8194], device='cuda:0')
Epoch 010 | Loss: 0.6004 | Train Acc: 0.6775 | Train PR-AUC: 0.4274 | Val Acc: 0.6817 | Val ROC-AUC: 0.8451 | Val PR-AUC: 0.3941
Saved best model.
Epoch 020 | Loss: 0.4745 | Train Acc: 0.7837 | Train PR-AUC: 0.5978 | Val Acc: 0.7877 | Val ROC-AUC: 0.9077 | Val PR-AUC: 0.5708
Saved best model.
Epoch 030 | Loss: 0.3945 | Train Acc: 0.8566 | Train PR-AUC: 0.6493 | Val Acc: 0.8561 | Val ROC-AUC: 0.9392 | Val

,node_id,fraud_probability,predicted_label,address,known_label,is_labelled
1625542,1625542,1.000000,1,0xdf9aceddd8a8c130dfe3015c0b1b507cf6571fc9,1,True
1625536,1625536,1.000000,1,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,1,True
1625539,1625539,1.000000,1,0xdac17f958d2ee523a2206206994597c13d831ec7,1,True
1625559,1625559,1.000000,1,0x904567252d8f48555b7447c67dca23f0372e16be,1,True
1625585,1625585,1.000000,1,0xcf068d287b206f20e64e9259f44c4f7130be7297,1,True
1625813,1625813,0.999998,1,0xc3b7d4ada2af58e6dc7b4fb303a0de47ade894c9,-1,False
46,46,0.999998,1,0x1ab4973a48dc892cd9971ece8e01dcc7688f8f23,1,True
246,246,0.999996,1,0xa03400e098f4421b34a3a44a1b4e571419517687,1,True
1686679,1686679,0.999990,1,0xc4bc3b5e66c461c3a57b21c85763ad2306311630,1,True
2482,2482,0.999988,1,0xb70f98e53bee56a2e18a15bcd826579f495076f0,1,True


### Result comment: GraphSAGE embeddings plus Snap ML motif-feature MLP

The Snap ML late-fusion model also performs well, with ROC-AUC ≈ 0.980 and PR-AUC ≈ 0.867. Fraud precision is about 0.78 and recall about 0.81. This is a major improvement over the Snap motif-only models, confirming that Snap-derived motif features are useful when combined with GraphSAGE embeddings. However, it is slightly weaker than the temporal motif late-fusion model in recall and conceptual fit, so the temporal motif late-fusion architecture remains the preferred motif-based implementation.


## 20. Comparative reading of model results

Across the updated runs, the most important pattern is unchanged: motif features are useful as complementary signals, but not as standalone replacements for the basic node features and graph structure. The Snap ML motif-only models have the weakest PR-AUC values, around 0.428 for one hidden layer and 0.491 for two hidden layers, with fraud precision staying around 0.40. This shows that motif-only representations do not capture enough information to distinguish fraudulent accounts robustly.

The strongest practical model remains the validation-threshold GraphSAGE baseline. It reaches PR-AUC ≈ 0.911, fraud precision ≈ 0.82, fraud recall ≈ 0.92, and fraud F1 ≈ 0.87. The key reason is threshold tuning: the default GraphSAGE model already ranked fraud nodes well, but the validation-selected threshold substantially reduced false positives while preserving high recall.

The community-aware label setting remains important because it makes the task more realistic. Its lower performance is expected: the normal class is harder because it contains nodes that are structurally closer to fraud-related regions of the graph. This label set is therefore useful for stress-testing the model rather than for maximizing headline metrics.

For temporal motifs, the direct-input GraphSAGE models show that motifs add signal, especially when moving from one to two hidden layers. However, the late-fusion architecture makes better conceptual use of them. The GraphSAGE encoder + temporal motif MLP reaches PR-AUC ≈ 0.869, fraud precision ≈ 0.78, and fraud recall ≈ 0.86. This supports the interpretation that temporal motifs should be kept as explicit behavioural features and combined with GraphSAGE embeddings after message passing, rather than being treated only as ordinary input attributes.

Overall, the updated results support three conclusions: first, threshold tuning is essential for making GraphSAGE operationally useful; second, motif-only models are insufficient; third, temporal motifs become useful when combined with basic node features and topology-aware GraphSAGE embeddings.
